In [ ]:
# ==========================================================
# NOTEBOOK 04
# ESTATÍSTICAS HISTÓRICAS DO PAS/UNB
#
# Parte 1
# - Configuração
# - Leitura dos CSVs consolidados
# - Construção da base analítica
# - Validação básica
# ==========================================================

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

# ----------------------------------------------------------
# Configuração
# ----------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

INICIO = time.time()

# ==========================================================
# DIRETÓRIOS
# ==========================================================


def resolver_base() -> Path:
    """Resolve o diretório raiz do projeto em qualquer ambiente."""
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (candidate / "tsv").exists() and (candidate / "catalogos").exists():
            return candidate

    return cwd


PASTA_BASE = resolver_base()

PASTA_DIAGNOSTICOS = PASTA_BASE / "diagnosticos"
PASTA_SAIDA = PASTA_BASE / "estatisticas"

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

TRIENIOS = [
    "2018-2020",
    "2019-2021",
    "2020-2022",
    "2021-2023",
    "2022-2024",
    "2023-2025",
]

# ==========================================================
# FUNÇÕES AUXILIARES
# ==========================================================

def ler_csv(trienio):
    """
    Lê o CSV consolidado de um triênio e acrescenta
    informações temporais.
    """

    arquivo = PASTA_DIAGNOSTICOS / f"{trienio}.csv"

    df = pd.read_csv(arquivo)

    ano_inicial, ano_final = map(int, trienio.split("-"))

    df["trienio"] = trienio
    df["ano_inicial"] = ano_inicial
    df["ano_final"] = ano_final
    df["indice_trienio"] = TRIENIOS.index(trienio) + 1

    return df


def salvar_csv(df, nome):
    df.to_csv(
        PASTA_SAIDA / nome,
        index=False,
        encoding="utf-8-sig"
    )


def salvar_parquet(df, nome):
    df.to_parquet(
        PASTA_SAIDA / nome,
        index=False
    )


def salvar_json(obj, nome):
    with open(
        PASTA_SAIDA / nome,
        "w",
        encoding="utf8"
    ) as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=4
        )


# ==========================================================
# LEITURA DOS TRIÊNIOS
# ==========================================================

dfs = [ler_csv(t) for t in TRIENIOS]

df = (
    pd.concat(dfs, ignore_index=True)
      .rename(columns={
          "Campus": "campus",
          "Curso": "curso",
          "Turno": "turno",
          "Modalidade": "modalidade",
          "Nota": "nota"
      })
      .sort_values(
          ["indice_trienio", "campus", "curso"]
      )
      .reset_index(drop=True)
)



# ==========================================================
# COLUNAS ANALÍTICAS
# ==========================================================

df["curso_campus"] = (
    df["curso"] + " | " + df["campus"]
)

df["curso_turno"] = (
    df["curso"] + " | " + df["turno"]
)

df["curso_modalidade"] = (
    df["curso"] + " | " + df["modalidade"]
)

df["curso_campus_turno"] = (
    df["curso"]
    + " | "
    + df["campus"]
    + " | "
    + df["turno"]
)

# ==========================================================
# CHAVES ANALÍTICAS
# ==========================================================

df["curso_modalidade"] = (
    df["curso"]
    + " | "
    + df["modalidade_normalizada"]
)

df["curso_modalidade_campus"] = (
    df["curso"]
    + " | "
    + df["modalidade_normalizada"]
    + " | "
    + df["campus"]
)

df["curso_modalidade_turno"] = (
    df["curso"]
    + " | "
    + df["modalidade_normalizada"]
    + " | "
    + df["turno"]
)

# ==========================================================
# VALIDAÇÃO BÁSICA
# ==========================================================

duplicatas = df.duplicated().sum()

nulos = (
    df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

resumo = {
    "linhas": len(df),
    "colunas": len(df.columns),
    "duplicatas": int(duplicatas),
    "memoria_mb": round(
        df.memory_usage(deep=True).sum() / 1024**2,
        2
    ),
}

# ==========================================================
# EXPORTAÇÃO DA BASE ANALÍTICA
# ==========================================================

salvar_csv(df, "base_consolidada.csv")
salvar_parquet(df, "base_consolidada.parquet")

# ==========================================================
# RESUMO
# ==========================================================

print("=" * 60)
print("BASE ANALÍTICA CONSOLIDADA")
print("=" * 60)

print(f"Triênios ............ {len(TRIENIOS)}")
print(f"Registros ........... {len(df):,}")
print(f"Colunas ............. {len(df.columns)}")
print(f"Duplicatas .......... {duplicatas}")
print(f"Memória ............. {resumo['memoria_mb']} MB")

print("\nPrimeiras linhas:\n")
display(df.head())

print("\nValores ausentes:\n")
display(nulos[nulos > 0])

print("\nArquivos produzidos:")
print(" - base_consolidada.csv")
print(" - base_consolidada.parquet")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE ANALÍTICA CONSOLIDADA
Triênios ............ 6
Registros ........... 2,306
Colunas ............. 24
Duplicatas .......... 0
Memória ............. 3.32 MB

Primeiras linhas:



,Subprograma,Ano,campus,curso,turno,modalidade,modalidade_normalizada,escola_publica,faixa_renda,grupo_etnico,pcd,limite_salario_minimo,Nota Mínima,Nota Máxima,trienio,ano_inicial,ano_final,indice_trienio,curso_campus,curso_turno,curso_modalidade,curso_campus_turno,curso_modalidade_campus,curso_modalidade_turno
0,2018-2020,2020,Ceilândia,Enfermagem (Bacharelado),Diurno,EP_R1_PPI,EP_R1_PPI,True,R1,PPI,False,1.500,-50.128,18.233,2018-2020,2018,2020,1,Enfermagem (Bacharelado) | Ceilândia,Enfermagem (Bacharelado) | Diurno,Enfermagem (Bacharelado) | EP_R1_PPI,Enfermagem (Bacharelado) | Ceilândia | Diurno,Enfermagem (Bacharelado) | EP_R1_PPI | Ceilândia,Enfermagem (Bacharelado) | EP_R1_PPI | Diurno
1,2018-2020,2020,Ceilândia,Enfermagem (Bacharelado),Diurno,EP_R1_NPPI,EP_R1_NPPI,True,R1,NPPI,False,1.500,-16.360,12.216,2018-2020,2018,2020,1,Enfermagem (Bacharelado) | Ceilândia,Enfermagem (Bacharelado) | Diurno,Enfermagem (Bacharelado) | EP_R1_NPPI,Enfermagem (Bacharelado) | Ceilândia | Diurno,Enfermagem (Bacharelado) | EP_R1_NPPI | Ceilândia,Enfermagem (Bacharelado) | EP_R1_NPPI | Diurno
2,2018-2020,2020,Ceilândia,Enfermagem (Bacharelado),Diurno,EP_R2_PPI,EP_R2_PPI,True,R2,PPI,False,1.500,-20.508,11.174,2018-2020,2018,2020,1,Enfermagem (Bacharelado) | Ceilândia,Enfermagem (Bacharelado) | Diurno,Enfermagem (Bacharelado) | EP_R2_PPI,Enfermagem (Bacharelado) | Ceilândia | Diurno,Enfermagem (Bacharelado) | EP_R2_PPI | Ceilândia,Enfermagem (Bacharelado) | EP_R2_PPI | Diurno
3,2018-2020,2020,Ceilândia,Enfermagem (Bacharelado),Diurno,EP_R2_NPPI,EP_R2_NPPI,True,R2,NPPI,False,1.500,12.110,43.716,2018-2020,2018,2020,1,Enfermagem (Bacharelado) | Ceilândia,Enfermagem (Bacharelado) | Diurno,Enfermagem (Bacharelado) | EP_R2_NPPI,Enfermagem (Bacharelado) | Ceilândia | Diurno,Enfermagem (Bacharelado) | EP_R2_NPPI | Ceilândia,Enfermagem (Bacharelado) | EP_R2_NPPI | Diurno
4,2018-2020,2020,Ceilândia,Enfermagem (Bacharelado),Diurno,CN,CN,False,NaN,NEGROS,False,NaN,6.016,17.233,2018-2020,2018,2020,1,Enfermagem (Bacharelado) | Ceilândia,Enfermagem (Bacharelado) | Diurno,Enfermagem (Bacharelado) | CN,Enfermagem (Bacharelado) | Ceilândia | Diurno,Enfermagem (Bacharelado) | CN | Ceilândia,Enfermagem (Bacharelado) | CN | Diurno



Valores ausentes:



,0
limite_salario_minimo,909
faixa_renda,909
grupo_etnico,534
Nota Máxima,26
Nota Mínima,5



Arquivos produzidos:
 - base_consolidada.csv
 - base_consolidada.parquet


In [192]:
# ==========================================================
# PARTE 2
# ESTATÍSTICAS DESCRITIVAS
# ==========================================================

# ==========================================================
# FUNÇÕES
# ==========================================================

def resumo_agrupado(df, grupos):

    if isinstance(grupos, str):
        grupos = [grupos]

    resumo = (

        df

        .groupby(grupos, dropna=False)

        .agg(

            registros=("curso", "count"),

            nota_min_media=("Nota Mínima", "mean"),
            nota_min_mediana=("Nota Mínima", "median"),
            nota_min_dp=("Nota Mínima", "std"),
            nota_min_min=("Nota Mínima", "min"),
            nota_min_max=("Nota Mínima", "max"),

            nota_max_media=("Nota Máxima", "mean"),
            nota_max_mediana=("Nota Máxima", "median"),
            nota_max_dp=("Nota Máxima", "std"),
            nota_max_min=("Nota Máxima", "min"),
            nota_max_max=("Nota Máxima", "max"),

        )

        .reset_index()

    )

    amplitude = (

        df

        .assign(
            amplitude=lambda x:
            x["Nota Máxima"] - x["Nota Mínima"]
        )

        .groupby(grupos, dropna=False)

        .agg(

            amplitude_media=("amplitude", "mean"),
            amplitude_dp=("amplitude", "std"),
            amplitude_min=("amplitude", "min"),
            amplitude_max=("amplitude", "max"),

        )

        .reset_index()

    )

    resumo = resumo.merge(
        amplitude,
        on=grupos,
        how="left"
    )

    resumo["coef_var_min"] = (
        resumo["nota_min_dp"]
        / resumo["nota_min_media"]
        * 100
    )

    resumo["coef_var_max"] = (
        resumo["nota_max_dp"]
        / resumo["nota_max_media"]
        * 100
    )

    return resumo.round(3)


# ----------------------------------------------------------

def resumo_trienios(df):

    return (

        df

        .groupby("trienio")

        .agg(

            registros=("curso", "count"),

            cursos=("curso", "nunique"),

            campi=("campus", "nunique"),

            modalidades=("modalidade_normalizada", "nunique"),

            media_min=("Nota Mínima", "mean"),

            media_max=("Nota Máxima", "mean"),

            menor_nota=("Nota Mínima", "min"),

            maior_nota=("Nota Máxima", "max"),

        )

        .reset_index()

        .round(3)

    )


# ==========================================================
# ESTATÍSTICAS GERAIS
# ==========================================================

estatisticas_gerais = {

    "registros": len(df),

    "triênios": df["trienio"].nunique(),

    "cursos": df["curso"].nunique(),

    "campi": df["campus"].nunique(),

    "modalidades": df["modalidade_normalizada"].nunique(),

    "nota_minima_global": round(
        df["Nota Mínima"].min(), 3
    ),

    "nota_maxima_global": round(
        df["Nota Máxima"].max(), 3
    ),

    "media_nota_minima": round(
        df["Nota Mínima"].mean(), 3
    ),

    "media_nota_maxima": round(
        df["Nota Máxima"].mean(), 3
    ),

    "amplitude_media": round(
        (
            df["Nota Máxima"]
            - df["Nota Mínima"]
        ).mean(),
        3
    )

}

salvar_json(
    estatisticas_gerais,
    "estatisticas_gerais.json"
)

# ==========================================================
# TABELAS
# ==========================================================

estat_trienios = resumo_trienios(df)

estat_cursos = resumo_agrupado(
    df,
    "curso"
)

estat_curso_modalidade = resumo_agrupado(
    df,
    ["curso", "modalidade_normalizada"]
)

estat_campi = resumo_agrupado(
    df,
    "campus"
)

estat_turnos = resumo_agrupado(
    df,
    "turno"
)

estat_modalidades = resumo_agrupado(
    df,
    "modalidade_normalizada"
)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

salvar_csv(
    estat_trienios,
    "trienios.csv"
)

salvar_csv(
    estat_cursos,
    "cursos.csv"
)

salvar_csv(
    estat_curso_modalidade,
    "curso_modalidade.csv"
)

salvar_csv(
    estat_campi,
    "campi.csv"
)

salvar_csv(
    estat_turnos,
    "turnos.csv"
)

salvar_csv(
    estat_modalidades,
    "modalidades.csv"
)

# ==========================================================
# RELATÓRIO
# ==========================================================

print("=" * 60)
print("ESTATÍSTICAS DESCRITIVAS")
print("=" * 60)

for chave, valor in estatisticas_gerais.items():
    print(f"{chave:<25} {valor}")

print("\nResumo por triênio")
display(estat_trienios)

print("\nResumo por curso × modalidade")
display(estat_curso_modalidade.head())

print("\nArquivos produzidos")

for arquivo in [
    "estatisticas_gerais.json",
    "trienios.csv",
    "cursos.csv",
    "curso_modalidade.csv",
    "campi.csv",
    "turnos.csv",
    "modalidades.csv",
]:
    print(f" ✓ {arquivo}")

ESTATÍSTICAS DESCRITIVAS
registros                 2306
triênios                  6
cursos                    90
campi                     4
modalidades               17
nota_minima_global        -105.487
nota_maxima_global        211.738
media_nota_minima         -12.719
media_nota_maxima         21.106
amplitude_media           34.6

Resumo por triênio


,trienio,registros,cursos,campi,modalidades,media_min,media_max,menor_nota,maior_nota
0,2018-2020,436,88,4,8,-29.419,17.616,-102.234,190.742
1,2019-2021,463,85,4,8,-10.243,15.816,-98.760,192.190
2,2020-2022,449,88,4,8,-8.834,26.814,-105.487,209.070
3,2021-2023,445,86,4,9,-7.308,24.771,-96.102,211.738
4,2022-2024,168,48,3,7,-10.902,9.029,-105.030,173.057
5,2023-2025,345,85,4,9,-7.823,25.942,-100.426,190.334



Resumo por curso × modalidade


,curso,modalidade_normalizada,registros,nota_min_media,nota_min_mediana,nota_min_dp,nota_min_min,nota_min_max,nota_max_media,nota_max_mediana,nota_max_dp,nota_max_min,nota_max_max,amplitude_media,amplitude_dp,amplitude_min,amplitude_max,coef_var_min,coef_var_max
0,Administração (Bacharelado),AC,12,-5.569,-4.295,47.289,-98.900,46.485,61.500,60.519,45.276,-40.646,120.300,67.069,31.228,12.670,137.886,-849.169,73.620
1,Administração (Bacharelado),CN,11,-25.820,-12.580,23.425,-54.846,-1.176,-7.046,1.212,31.035,-53.830,44.699,18.773,23.734,0.000,67.953,-90.726,-440.458
2,Administração (Bacharelado),EP_R1_NPPI,8,-47.563,-43.716,23.474,-90.624,-15.383,-5.380,-1.766,17.656,-34.780,16.153,42.184,26.446,0.000,87.100,-49.354,-328.203
3,Administração (Bacharelado),EP_R1_NPPIQ,1,-67.831,-67.831,NaN,-67.831,-67.831,-19.227,-19.227,NaN,-19.227,-19.227,48.604,NaN,48.604,48.604,NaN,NaN
4,Administração (Bacharelado),EP_R1_PPI,7,-41.041,-46.477,22.926,-72.089,2.342,0.366,-1.110,14.967,-16.586,23.576,41.407,23.039,0.000,62.919,-55.861,4094.216



Arquivos produzidos
 ✓ estatisticas_gerais.json
 ✓ trienios.csv
 ✓ cursos.csv
 ✓ curso_modalidade.csv
 ✓ campi.csv
 ✓ turnos.csv
 ✓ modalidades.csv


In [193]:
# ==========================================================
# PARTE 3A
# DATASET DE SÉRIES HISTÓRICAS
# ==========================================================

# ==========================================================
# COLUNAS DA SÉRIE
# ==========================================================

COLUNAS_SERIE = [

    "trienio",
    "indice_trienio",
    "Ano",

    "campus",
    "curso",
    "turno",

    "modalidade",
    "modalidade_normalizada",

    "escola_publica",
    "faixa_renda",
    "grupo_etnico",
    "pcd",
    "limite_salario_minimo",

    "Nota Mínima",
    "Nota Máxima",

]

# ==========================================================
# CONSTRUÇÃO
# ==========================================================

series = (

    df

    [COLUNAS_SERIE]

    .sort_values(

        [

            "curso",
            "modalidade_normalizada",
            "campus",
            "turno",
            "indice_trienio"

        ]

    )

    .reset_index(drop=True)

)

# ==========================================================
# IDENTIFICADORES ANALÍTICOS
# ==========================================================

series["id_curso"] = (
    series["curso"]
)

series["id_curso_campus"] = (
    series["curso"]
    + " | "
    + series["campus"]
)

series["id_curso_turno"] = (
    series["curso"]
    + " | "
    + series["turno"]
)

series["id_curso_modalidade"] = (
    series["curso"]
    + " | "
    + series["modalidade_normalizada"]
)

series["id_curso_campus_turno"] = (
    series["curso"]
    + " | "
    + series["campus"]
    + " | "
    + series["turno"]
)

# ----------------------------------------------------------
# Oferta completa de ingresso
# ----------------------------------------------------------

series["id_oferta"] = (

    series["curso"]

    + " | "

    + series["campus"]

    + " | "

    + series["turno"]

    + " | "

    + series["modalidade_normalizada"]

    + " | EP="

    + series["escola_publica"].fillna("NA").astype(str)

    + " | R="

    + series["faixa_renda"].fillna("NA").astype(str)

    + " | GE="

    + series["grupo_etnico"].fillna("NA").astype(str)

    + " | PCD="

    + series["pcd"].fillna("NA").astype(str)

    + " | SM="

    + series["limite_salario_minimo"].fillna("NA").astype(str)

)

# ==========================================================
# IDENTIFICADOR NUMÉRICO
# ==========================================================

series["id_oferta_num"] = (
    pd.factorize(series["id_oferta"])[0] + 1
)

# ==========================================================
# AMPLITUDE
# ==========================================================

series["amplitude"] = (

    series["Nota Máxima"]

    - series["Nota Mínima"]

)

# ==========================================================
# VARIAÇÃO ENTRE TRIÊNIOS
# ==========================================================

series["delta_min"] = (

    series

    .groupby("id_oferta")["Nota Mínima"]

    .diff()

)

series["delta_max"] = (

    series

    .groupby("id_oferta")["Nota Máxima"]

    .diff()

)

series["delta_amplitude"] = (

    series

    .groupby("id_oferta")["amplitude"]

    .diff()

)

# ==========================================================
# POSIÇÃO NA SÉRIE
# ==========================================================

series["ordem"] = (

    series

    .groupby("id_oferta")

    .cumcount()

    + 1

)

series["total_trienios"] = (

    series

    .groupby("id_oferta")["trienio"]

    .transform("count")

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

salvar_csv(

    series,

    "series_historicas.csv"

)

salvar_parquet(

    series,

    "series_historicas.parquet"

)

# ==========================================================
# RELATÓRIO
# ==========================================================

print("=" * 60)
print("DATASET DE SÉRIES HISTÓRICAS")
print("=" * 60)

print(f"Registros ............... {len(series):,}")

print(f"Cursos ................. {series['curso'].nunique()}")

print(f"Modalidades ............ {series['modalidade_normalizada'].nunique()}")

print(f"Ofertas ................ {series['id_oferta'].nunique()}")

print(f"Triênios ............... {series['trienio'].nunique()}")

print("\nPrimeiras observações")

display(series.head())

print("\nArquivos produzidos")

print(" ✓ series_historicas.csv")
print(" ✓ series_historicas.parquet")

DATASET DE SÉRIES HISTÓRICAS
Registros ............... 2,306
Cursos ................. 90
Modalidades ............ 17
Ofertas ................ 765
Triênios ............... 6

Primeiras observações


,trienio,indice_trienio,Ano,campus,curso,turno,modalidade,modalidade_normalizada,escola_publica,faixa_renda,grupo_etnico,pcd,limite_salario_minimo,Nota Mínima,Nota Máxima,id_curso,id_curso_campus,id_curso_turno,id_curso_modalidade,id_curso_campus_turno,id_oferta,id_oferta_num,amplitude,delta_min,delta_max,delta_amplitude,ordem,total_trienios
0,2018-2020,1,2020,Darcy Ribeiro,Administração (Bacharelado),Diurno,AC,AC,False,NaN,NaN,False,NaN,-5.813,69.035,Administração (Bacharelado),Administração (Bacharelado) | Darcy Ribeiro,Administração (Bacharelado) | Diurno,Administração (Bacharelado) | AC,Administração (Bacharelado) | Darcy Ribeiro | ...,Administração (Bacharelado) | Darcy Ribeiro | ...,1,74.848,NaN,NaN,NaN,1,6
1,2019-2021,2,2021,Darcy Ribeiro,Administração (Bacharelado),Diurno,AC,AC,False,NaN,NaN,False,NaN,36.705,120.300,Administração (Bacharelado),Administração (Bacharelado) | Darcy Ribeiro,Administração (Bacharelado) | Diurno,Administração (Bacharelado) | AC,Administração (Bacharelado) | Darcy Ribeiro | ...,Administração (Bacharelado) | Darcy Ribeiro | ...,1,83.595,42.518,51.265,8.747,2,6
2,2020-2022,3,2022,Darcy Ribeiro,Administração (Bacharelado),Diurno,AC,AC,False,NaN,NaN,False,NaN,45.565,98.219,Administração (Bacharelado),Administração (Bacharelado) | Darcy Ribeiro,Administração (Bacharelado) | Diurno,Administração (Bacharelado) | AC,Administração (Bacharelado) | Darcy Ribeiro | ...,Administração (Bacharelado) | Darcy Ribeiro | ...,1,52.654,8.860,-22.081,-30.941,3,6
3,2021-2023,4,2023,Darcy Ribeiro,Administração (Bacharelado),Diurno,AC,AC,False,NaN,NaN,False,NaN,46.485,94.873,Administração (Bacharelado),Administração (Bacharelado) | Darcy Ribeiro,Administração (Bacharelado) | Diurno,Administração (Bacharelado) | AC,Administração (Bacharelado) | Darcy Ribeiro | ...,Administração (Bacharelado) | Darcy Ribeiro | ...,1,48.388,0.920,-3.346,-4.266,4,6
4,2022-2024,5,2024,Darcy Ribeiro,Administração (Bacharelado),Diurno,AC,AC,False,NaN,NaN,False,NaN,0.137,12.807,Administração (Bacharelado),Administração (Bacharelado) | Darcy Ribeiro,Administração (Bacharelado) | Diurno,Administração (Bacharelado) | AC,Administração (Bacharelado) | Darcy Ribeiro | ...,Administração (Bacharelado) | Darcy Ribeiro | ...,1,12.670,-46.348,-82.066,-35.718,5,6



Arquivos produzidos
 ✓ series_historicas.csv
 ✓ series_historicas.parquet


In [194]:
# ==========================================================
# PARTE 3B
# INDICADORES HISTÓRICOS
# ==========================================================

def construir_indicadores(series):

    registros = []

    for id_oferta, g in series.groupby("id_oferta"):

        g = g.sort_values("indice_trienio")

        nota_min = g["Nota Mínima"].to_numpy()
        nota_max = g["Nota Máxima"].to_numpy()
        amplitude = g["amplitude"].to_numpy()

        n = len(g)

        primeira_min = nota_min[0]
        ultima_min = nota_min[-1]

        primeira_max = nota_max[0]
        ultima_max = nota_max[-1]

        delta_min = ultima_min - primeira_min
        delta_max = ultima_max - primeira_max

        delta_min_pct = (
            delta_min / primeira_min * 100
            if primeira_min != 0 else np.nan
        )

        delta_max_pct = (
            delta_max / primeira_max * 100
            if primeira_max != 0 else np.nan
        )

        velocidade_min = (
            delta_min / (n - 1)
            if n > 1 else np.nan
        )

        velocidade_max = (
            delta_max / (n - 1)
            if n > 1 else np.nan
        )

        registros.append({

            "id_oferta": id_oferta,

            "id_oferta_num":
                g["id_oferta_num"].iloc[0],

            "curso":
                g["curso"].iloc[0],

            "campus":
                g["campus"].iloc[0],

            "turno":
                g["turno"].iloc[0],

            "modalidade_normalizada":
                g["modalidade_normalizada"].iloc[0],

            "observacoes":
                n,

            "primeira_nota_min":
                primeira_min,

            "ultima_nota_min":
                ultima_min,

            "primeira_nota_max":
                primeira_max,

            "ultima_nota_max":
                ultima_max,

            "delta_min":
                delta_min,

            "delta_max":
                delta_max,

            "delta_min_pct":
                delta_min_pct,

            "delta_max_pct":
                delta_max_pct,

            "velocidade_min":
                velocidade_min,

            "velocidade_max":
                velocidade_max,

            "media_min":
                nota_min.mean(),

            "media_max":
                nota_max.mean(),

            "desvio_min":
                np.std(nota_min, ddof=1)
                if n > 1 else np.nan,

            "desvio_max":
                np.std(nota_max, ddof=1)
                if n > 1 else np.nan,

            "coef_var_min":
                (
                    np.std(nota_min, ddof=1)
                    / np.mean(nota_min)
                    * 100
                )
                if n > 1 else np.nan,

            "coef_var_max":
                (
                    np.std(nota_max, ddof=1)
                    / np.mean(nota_max)
                    * 100
                )
                if n > 1 else np.nan,

            "amplitude_media":
                amplitude.mean(),

            "amplitude_min":
                amplitude.min(),

            "amplitude_max":
                amplitude.max(),

        })

    return (
        pd.DataFrame(registros)
        .round(3)
        .sort_values("id_oferta_num")
        .reset_index(drop=True)
    )


# ==========================================================
# CONSTRUÇÃO
# ==========================================================

indicadores = construir_indicadores(series)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

salvar_csv(
    indicadores,
    "indicadores_historicos.csv"
)

salvar_parquet(
    indicadores,
    "indicadores_historicos.parquet"
)

# ==========================================================
# RELATÓRIO
# ==========================================================

print("=" * 60)
print("INDICADORES HISTÓRICOS")
print("=" * 60)

print(f"Ofertas analisadas ......... {len(indicadores):,}")

display(indicadores.head())

INDICADORES HISTÓRICOS
Ofertas analisadas ......... 765


,id_oferta,id_oferta_num,curso,campus,turno,modalidade_normalizada,observacoes,primeira_nota_min,ultima_nota_min,primeira_nota_max,ultima_nota_max,delta_min,delta_max,delta_min_pct,delta_max_pct,velocidade_min,velocidade_max,media_min,media_max,desvio_min,desvio_max,coef_var_min,coef_var_max,amplitude_media,amplitude_min,amplitude_max
0,Administração (Bacharelado) | Darcy Ribeiro | ...,1,Administração (Bacharelado),Darcy Ribeiro,Diurno,AC,6,-5.813,38.131,69.035,99.156,43.944,30.121,-755.961,43.631,8.789,6.024,26.868,82.398,23.412,37.802,87.136,45.877,55.530,12.670,83.595
1,Administração (Bacharelado) | Darcy Ribeiro | ...,2,Administração (Bacharelado),Darcy Ribeiro,Noturno,AC,6,-98.900,-7.050,38.986,44.901,91.850,5.915,-92.872,15.172,18.370,1.183,-38.006,40.601,42.970,45.085,-113.060,111.043,78.608,46.121,137.886
2,Administração (Bacharelado) | Darcy Ribeiro | ...,3,Administração (Bacharelado),Darcy Ribeiro,Diurno,CN,6,-40.808,-12.580,3.022,6.842,28.228,3.820,-69.173,126.406,5.646,0.764,-28.120,-7.373,22.874,36.280,-81.344,-492.069,20.747,0.000,49.680
3,Administração (Bacharelado) | Darcy Ribeiro | ...,4,Administração (Bacharelado),Darcy Ribeiro,Noturno,CN,5,-3.806,-54.846,1.212,-53.830,-51.040,-55.042,1341.040,-4541.419,-12.760,-13.760,-23.060,-6.654,26.464,27.610,-114.765,-414.944,16.406,1.016,67.953
4,Administração (Bacharelado) | Darcy Ribeiro | ...,5,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,4,-90.624,-59.221,-3.524,-21.197,31.403,-17.673,-34.652,501.504,10.468,-5.891,-51.749,-2.144,31.566,15.322,-60.997,-714.546,49.605,15.374,87.100


In [195]:
# ==========================================================
# PARTE 3C
# CONSOLIDAÇÕES ANALÍTICAS
# ==========================================================

def consolidar_indicadores(df, grupos):

    if isinstance(grupos, str):
        grupos = [grupos]

    resumo = (

        df

        .groupby(grupos, dropna=False)

        .agg(

            ofertas=("id_oferta_num", "count"),

            observacoes=("observacoes", "sum"),

            delta_min_medio=("delta_min", "mean"),
            delta_max_medio=("delta_max", "mean"),

            delta_min_pct_medio=("delta_min_pct", "mean"),
            delta_max_pct_medio=("delta_max_pct", "mean"),

            velocidade_min_media=("velocidade_min", "mean"),
            velocidade_max_media=("velocidade_max", "mean"),

            coef_var_min_medio=("coef_var_min", "mean"),
            coef_var_max_medio=("coef_var_max", "mean"),

            amplitude_media=("amplitude_media", "mean"),

            maior_crescimento=("delta_max", "max"),

            maior_queda=("delta_max", "min"),

        )

        .round(3)

        .reset_index()

    )

    return resumo

In [196]:
# ==========================================================
# PARTE 3D
# CLASSIFICAÇÃO DAS TENDÊNCIAS
# ==========================================================

def classificar_tendencias(series):

    registros = []

    for id_oferta, g in series.groupby("id_oferta"):

        g = g.sort_values("indice_trienio")

        nota_min = g["Nota Mínima"].to_numpy()
        nota_max = g["Nota Máxima"].to_numpy()

        if len(g) < 2:
            continue

        # --------------------------------------------------
        # Variações entre triênios
        # --------------------------------------------------

        delta_min = np.diff(nota_min)
        delta_max = np.diff(nota_max)

        velocidade_min = delta_min.mean()
        velocidade_max = delta_max.mean()

        # --------------------------------------------------
        # Consistência
        # --------------------------------------------------

        direcao = np.sign(delta_max)

        positivos = np.sum(direcao > 0)
        negativos = np.sum(direcao < 0)

        consistencia = (
            max(positivos, negativos)
            / len(direcao)
            * 100
        )

        # --------------------------------------------------
        # Oscilações
        # --------------------------------------------------

        mudancas = np.sum(
            direcao[:-1] != direcao[1:]
        )

        # --------------------------------------------------
        # Coeficientes de variação
        # --------------------------------------------------

        media_min = nota_min.mean()
        media_max = nota_max.mean()

        cv_min = (
            np.std(
                nota_min,
                ddof=1
            )
            / media_min
            * 100
        )

        cv_max = (
            np.std(
                nota_max,
                ddof=1
            )
            / media_max
            * 100
        )

        # --------------------------------------------------
        # Classe
        # --------------------------------------------------

        if (
            abs(velocidade_max) <= 0.5
            and cv_max < 5
        ):

            classe = "Estável"

        elif (
            velocidade_max > 0
            and consistencia >= 80
            and mudancas <= 1
        ):

            classe = "Crescimento"

        elif (
            velocidade_max < 0
            and consistencia >= 80
            and mudancas <= 1
        ):

            classe = "Queda"

        else:

            classe = "Oscilante"

        registros.append({

            "id_oferta":

                id_oferta,

            "id_oferta_num":

                g["id_oferta_num"].iloc[0],

            "curso":

                g["curso"].iloc[0],

            "campus":

                g["campus"].iloc[0],

            "turno":

                g["turno"].iloc[0],

            "modalidade_normalizada":

                g["modalidade_normalizada"].iloc[0],

            "classe":

                classe,

            "velocidade_min":

                round(
                    velocidade_min,
                    3
                ),

            "velocidade_max":

                round(
                    velocidade_max,
                    3
                ),

            "consistencia":

                round(
                    consistencia,
                    1
                ),

            "mudancas_direcao":

                int(
                    mudancas
                ),

            "coef_var_min":

                round(
                    cv_min,
                    3
                ),

            "coef_var_max":

                round(
                    cv_max,
                    3
                ),

        })

    return (

        pd.DataFrame(registros)

        .sort_values(
            "id_oferta_num"
        )

        .reset_index(drop=True)

    )


# ==========================================================
# CONSTRUÇÃO
# ==========================================================

tendencias = classificar_tendencias(
    series
)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

salvar_csv(
    tendencias,
    "tendencias_base.csv"
)

salvar_parquet(
    tendencias,
    "tendencias_base.parquet"
)

# ==========================================================
# RELATÓRIO
# ==========================================================

print("=" * 60)
print("CLASSIFICAÇÃO DAS TENDÊNCIAS")
print("=" * 60)

display(

    tendencias

    .groupby("classe")

    .size()

    .rename("quantidade")

    .to_frame()

)

CLASSIFICAÇÃO DAS TENDÊNCIAS


,quantidade
classe,
Crescimento,54
Estável,5
Oscilante,386
Queda,50


In [197]:
# ==========================================================
# PARTE 3E.1
# CONFIGURAÇÃO DA INFRAESTRUTURA PREDITIVA
# ==========================================================

from sklearn.linear_model import LinearRegression

# ==========================================================
# CONFIGURAÇÕES GERAIS
# ==========================================================

# Quantidade mínima de observações para ajuste
# dos modelos de regressão.

MIN_OBSERVACOES_REGRESSAO = 2

# Pesos utilizados pela média móvel ponderada.
# Os pesos crescem automaticamente conforme o
# número de triênios disponíveis.

def pesos_temporais(n):
    """
    Retorna pesos temporais crescentes.

    Exemplo:
        n = 6 -> [1,2,3,4,5,6]
        n = 3 -> [1,2,3]
    """

    return np.arange(
        1,
        n + 1,
        dtype=float
    )

# ==========================================================
# MODELOS DISPONÍVEIS
# ==========================================================

MODELOS_PREDITIVOS = [

    "persistencia",

    "regressao_linear",

    "regressao_ponderada",

    "media_movel",

]

# ==========================================================
# COLUNAS ANALISADAS
# ==========================================================

COLUNAS_NOTAS = [

    "Nota Mínima",

    "Nota Máxima",

]

# ==========================================================
# COLUNAS IDENTIFICADORAS
# ==========================================================

COLUNAS_IDENTIFICACAO = [

    "id_oferta",

    "id_oferta_num",

    "curso",

    "campus",

    "turno",

    "modalidade_normalizada",

]

# ==========================================================
# COLUNAS DA SÉRIE TEMPORAL
# ==========================================================

COLUNAS_SERIE = [

    "indice_trienio",

    "trienio",

    "Nota Mínima",

    "Nota Máxima",

]

In [198]:
# ==========================================================
# PARTE 3E.2
# PREPARAÇÃO DAS SÉRIES TEMPORAIS
# ==========================================================

# ==========================================================
# PREPARAÇÃO DE UMA SÉRIE
# ==========================================================

def preparar_serie(g, coluna):
    """
    Extrai uma série temporal removendo valores ausentes.
    """

    dados = (

        g[
            ["indice_trienio", coluna]
        ]

        .dropna()

        .sort_values("indice_trienio")

        .reset_index(drop=True)

    )

    x = (
        dados["indice_trienio"]
        .to_numpy(dtype=float)
        .reshape(-1, 1)
    )

    y = (
        dados[coluna]
        .to_numpy(dtype=float)
    )

    return x, y


# ==========================================================
# CONSTRUÇÃO DAS SÉRIES
# ==========================================================

def construir_series(series):
    """
    Organiza todas as séries temporais por oferta.
    """

    dict_series = {}

    for id_oferta, g in (

        series

        .sort_values("indice_trienio")

        .groupby("id_oferta")

    ):

        g = (

            g

            .sort_values("indice_trienio")

            .reset_index(drop=True)

        )

        x_min, y_min = preparar_serie(
            g,
            "Nota Mínima"
        )

        x_max, y_max = preparar_serie(
            g,
            "Nota Máxima"
        )

        dict_series[id_oferta] = {

            # ------------------------------------------
            # Identificação
            # ------------------------------------------

            "id_oferta":
                id_oferta,

            "id_oferta_num":
                g["id_oferta_num"].iat[0],

            "curso":
                g["curso"].iat[0],

            "campus":
                g["campus"].iat[0],

            "turno":
                g["turno"].iat[0],

            "modalidade":
                g["modalidade_normalizada"].iat[0],

            # ------------------------------------------
            # Dados completos
            # ------------------------------------------

            "dados":
                g.copy(),

            # ------------------------------------------
            # Séries
            # ------------------------------------------

            "x_min":
                x_min,

            "y_min":
                y_min,

            "x_max":
                x_max,

            "y_max":
                y_max,
            # ------------------------------------------
            # Pesos temporais
            # ------------------------------------------

            "pesos_min":
                pesos_temporais(len(y_min)),

            "pesos_max":
                pesos_temporais(len(y_max)),

            # ------------------------------------------
            # Quantidade de observações
            # ------------------------------------------

            "n_min":
                len(y_min),

            "n_max":
                len(y_max),

            "n":
                max(
                    len(y_min),
                    len(y_max)
                ),

            # ------------------------------------------
            # Validação
            # ------------------------------------------

            "possui_min":
                len(y_min) > 0,

            "possui_max":
                len(y_max) > 0,

            "modelo_min_valido":
                len(y_min) >= MIN_OBSERVACOES_REGRESSAO,

            "modelo_max_valido":
                len(y_max) >= MIN_OBSERVACOES_REGRESSAO,

            # ------------------------------------------
            # Histórico
            # ------------------------------------------

            "trienios":
                g["trienio"].tolist(),

            "indices":
                g["indice_trienio"].tolist(),

        }

    return dict_series


# ==========================================================
# EXECUÇÃO
# ==========================================================

dict_series = construir_series(series)

# ==========================================================
# TABELA RESUMO
# ==========================================================

estatisticas_series = pd.DataFrame([

    {

        "id_oferta":
            s["id_oferta"],

        "curso":
            s["curso"],

        "campus":
            s["campus"],

        "turno":
            s["turno"],

        "modalidade":
            s["modalidade"],

        "obs_min":
            s["n_min"],

        "obs_max":
            s["n_max"],

        "modelo_min":
            s["modelo_min_valido"],

        "modelo_max":
            s["modelo_max_valido"],

    }

    for s in dict_series.values()

])

# ==========================================================
# RELATÓRIO
# ==========================================================

print("=" * 60)
print("PREPARAÇÃO DAS SÉRIES")
print("=" * 60)

print(f"Ofertas..................... {len(dict_series)}")

print(
    f"Modelos mínimos válidos..... "
    f"{estatisticas_series['modelo_min'].sum()}"
)

print(
    f"Modelos máximos válidos..... "
    f"{estatisticas_series['modelo_max'].sum()}"
)

print()

display(estatisticas_series.head())

# ==========================================================
# EXEMPLO
# ==========================================================

exemplo = next(iter(dict_series.values()))

print("\nEXEMPLO\n")

print(f"Curso.............. {exemplo['curso']}")
print(f"Campus............. {exemplo['campus']}")
print(f"Turno.............. {exemplo['turno']}")
print(f"Modalidade......... {exemplo['modalidade']}")

print()

print("Triênios")

print(exemplo["trienios"])

print()

print("Notas mínimas")

print(exemplo["y_min"])

print()

print("Notas máximas")

print(exemplo["y_max"])

PREPARAÇÃO DAS SÉRIES
Ofertas..................... 765
Modelos mínimos válidos..... 495
Modelos máximos válidos..... 495



,id_oferta,curso,campus,turno,modalidade,obs_min,obs_max,modelo_min,modelo_max
0,Administração (Bacharelado) | Darcy Ribeiro | ...,Administração (Bacharelado),Darcy Ribeiro,Diurno,AC,6,6,True,True
1,Administração (Bacharelado) | Darcy Ribeiro | ...,Administração (Bacharelado),Darcy Ribeiro,Diurno,CN,6,6,True,True
2,Administração (Bacharelado) | Darcy Ribeiro | ...,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,4,4,True,True
3,Administração (Bacharelado) | Darcy Ribeiro | ...,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPIQ,1,1,False,False
4,Administração (Bacharelado) | Darcy Ribeiro | ...,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_PPI,4,4,True,True



EXEMPLO

Curso.............. Administração (Bacharelado)
Campus............. Darcy Ribeiro
Turno.............. Diurno
Modalidade......... AC

Triênios
['2018-2020', '2019-2021', '2020-2022', '2021-2023', '2022-2024', '2023-2025']

Notas mínimas
[-5.813 36.705 45.565 46.485  0.137 38.131]

Notas máximas
[ 69.035 120.3    98.219  94.873  12.807  99.156]


In [199]:
# ==========================================================
# PARTE 3E.3
# MÉTRICAS DAS SÉRIES TEMPORAIS
# ==========================================================

# ==========================================================
# MÉTRICAS DE UMA SÉRIE
# ==========================================================

def metricas_serie(y):
    """
    Calcula métricas descritivas de uma série temporal.
    """

    y = np.asarray(y, dtype=float)

    n = len(y)

    if n == 0:

        return {

            "n": 0,

            "media": np.nan,

            "mediana": np.nan,

            "minimo": np.nan,

            "maximo": np.nan,

            "amplitude": np.nan,

            "desvio": np.nan,

            "coef_var": np.nan,

            "consistencia": np.nan,

            "mudancas": np.nan,

            "volatilidade": np.nan,

        }

    media = float(np.mean(y))

    mediana = float(np.median(y))

    minimo = float(np.min(y))

    maximo = float(np.max(y))

    amplitude = maximo - minimo

    # ------------------------------------------------------
    # Apenas uma observação
    # ------------------------------------------------------

    if n == 1:

        return {

            "n": 1,

            "media": media,

            "mediana": mediana,

            "minimo": minimo,

            "maximo": maximo,

            "amplitude": 0.0,

            "desvio": 0.0,

            "coef_var": 0.0,

            "consistencia": 100.0,

            "mudancas": 0,

            "volatilidade": 0.0,

        }

    # ------------------------------------------------------
    # Duas ou mais observações
    # ------------------------------------------------------

    desvio = float(
        np.std(
            y,
            ddof=1
        )
    )

    coef_var = (

        desvio

        / media

        * 100

        if media != 0

        else np.nan

    )

    deltas = np.diff(y)

    sinais = np.sign(deltas)

    positivos = np.sum(sinais > 0)

    negativos = np.sum(sinais < 0)

    consistencia = (

        max(
            positivos,
            negativos
        )

        /

        len(sinais)

        * 100

    )

    mudancas = int(

        np.sum(

            sinais[:-1]

            !=

            sinais[1:]

        )

    )

    volatilidade = (

        float(

            np.std(
                deltas,
                ddof=1
            )

        )

        if len(deltas) > 1

        else 0.0

    )

    return {

        "n": n,

        "media": media,

        "mediana": mediana,

        "minimo": minimo,

        "maximo": maximo,

        "amplitude": amplitude,

        "desvio": desvio,

        "coef_var": coef_var,

        "consistencia": consistencia,

        "mudancas": mudancas,

        "volatilidade": volatilidade,

    }


# ==========================================================
# TESTE
# ==========================================================

exemplo = next(iter(dict_series.values()))

print("=" * 60)
print("MÉTRICAS DAS SÉRIES")
print("=" * 60)

metricas_min = metricas_serie(
    exemplo["y_min"]
)

metricas_max = metricas_serie(
    exemplo["y_max"]
)

teste_metricas = pd.DataFrame({

    "Métrica": metricas_min.keys(),

    "Nota mínima": metricas_min.values(),

    "Nota máxima": metricas_max.values(),

})

display(teste_metricas)

MÉTRICAS DAS SÉRIES


,Métrica,Nota mínima,Nota máxima
0,n,6.000,6.000
1,media,26.868,82.398
2,mediana,37.418,96.546
3,minimo,-5.813,12.807
4,maximo,46.485,120.300
5,amplitude,52.298,107.493
6,desvio,23.412,37.802
7,coef_var,87.136,45.877
8,consistencia,80.000,60.000
9,mudancas,2.000,2.000


In [200]:
# ==========================================================
# PARTE 3E.4
# MODELOS PREDITIVOS
# ==========================================================

from sklearn.linear_model import LinearRegression

# ==========================================================
# FUNÇÃO AUXILIAR
# ==========================================================

def serie_info(y):
    """
    Resume a série para auxiliar os modelos.
    """

    y = np.asarray(y, dtype=float)

    return {

        "n": len(y),

        "tem_dados": len(y) > 0,

        "permite_regressao":
            len(y) >= MIN_OBSERVACOES_REGRESSAO,

    }


# ==========================================================
# MODELO 1
# PERSISTÊNCIA
# ==========================================================

def modelo_persistencia(x, y):

    info = serie_info(y)

    if not info["tem_dados"]:
        return None

    return {

        "modelo": "Persistência",

        "previsao": float(y[-1]),

        "r2": np.nan,

        "coef_angular": np.nan,

        "intercepto": np.nan,

    }


# ==========================================================
# MODELO 2
# REGRESSÃO LINEAR
# ==========================================================

def modelo_regressao_linear(x, y):

    info = serie_info(y)

    if not info["permite_regressao"]:
        return None

    modelo = LinearRegression()

    modelo.fit(x, y)

    proximo = float(x.max() + 1)

    return {

        "modelo": "Regressão Linear",

        "previsao":
            float(
                modelo.predict([[proximo]])[0]
            ),

        "r2":
            float(
                modelo.score(x, y)
            ),

        "coef_angular":
            float(modelo.coef_[0]),

        "intercepto":
            float(modelo.intercept_),

    }


# ==========================================================
# MODELO 3
# REGRESSÃO PONDERADA
# ==========================================================

def modelo_regressao_ponderada(x, y):

    info = serie_info(y)

    if not info["permite_regressao"]:
        return None

    pesos = pesos_temporais(len(y))

    modelo = LinearRegression()

    modelo.fit(
        x,
        y,
        sample_weight=pesos
    )

    proximo = float(x.max() + 1)

    return {

        "modelo": "Regressão Ponderada",

        "previsao":
            float(
                modelo.predict([[proximo]])[0]
            ),

        "r2":
            float(
                modelo.score(x, y)
            ),

        "coef_angular":
            float(modelo.coef_[0]),

        "intercepto":
            float(modelo.intercept_),

    }


# ==========================================================
# MODELO 4
# MÉDIA MÓVEL PONDERADA
# ==========================================================

def modelo_media_movel(x, y):

    info = serie_info(y)

    if not info["tem_dados"]:
        return None

    pesos = pesos_temporais(len(y))

    return {

        "modelo": "Média Móvel",

        "previsao":
            float(
                np.average(
                    y,
                    weights=pesos
                )
            ),

        "r2": np.nan,

        "coef_angular": np.nan,

        "intercepto": np.nan,

    }


# ==========================================================
# REGISTRO DOS MODELOS
# ==========================================================

MODELOS = {

    "persistencia":
        modelo_persistencia,

    "regressao_linear":
        modelo_regressao_linear,

    "regressao_ponderada":
        modelo_regressao_ponderada,

    "media_movel":
        modelo_media_movel,

}


# ==========================================================
# TESTE
# ==========================================================

exemplo = next(

    oferta

    for oferta in dict_series.values()

    if len(oferta["y_max"]) >= 2

)

resultado = []

for nome, funcao in MODELOS.items():

    r = funcao(

        exemplo["x_max"],

        exemplo["y_max"]

    )

    if r is not None:

        resultado.append(r)

print("=" * 60)
print("MODELOS PREDITIVOS")
print("=" * 60)

display(pd.DataFrame(resultado))

MODELOS PREDITIVOS


,modelo,previsao,r2,coef_angular,intercepto
0,Persistência,99.156,NaN,NaN,NaN
1,Regressão Linear,64.876,0.061,-5.006,99.920
2,Regressão Ponderada,62.418,0.059,-5.928,103.914
3,Média Móvel,78.226,NaN,NaN,NaN


In [201]:
# ==========================================================
# PARTE 3E.5
# EXECUTOR DOS MODELOS
# ==========================================================

def executar_modelos(dict_series):
    """
    Executa todos os modelos preditivos, calcula as métricas
    das séries históricas e consolida o ensemble.

    Todo o resultado fica armazenado dentro de dict_series,
    tornando-o a fonte única de verdade da infraestrutura.
    """

    for oferta in dict_series.values():

        # ==================================================
        # MÉTRICAS DAS SÉRIES
        # ==================================================

        oferta["metricas"] = {

            "min": metricas_serie(
                oferta["y_min"]
            ),

            "max": metricas_serie(
                oferta["y_max"]
            ),

        }

        # ==================================================
        # MODELOS
        # ==================================================

        oferta["modelos"] = {

            "min": {},

            "max": {},

        }

        # --------------------------------------------------
        # Nota mínima
        # --------------------------------------------------

        for nome, funcao in MODELOS.items():

            oferta["modelos"]["min"][nome] = funcao(

                oferta["x_min"],

                oferta["y_min"]

            )

        # --------------------------------------------------
        # Nota máxima
        # --------------------------------------------------

        for nome, funcao in MODELOS.items():

            oferta["modelos"]["max"][nome] = funcao(

                oferta["x_max"],

                oferta["y_max"]

            )

        # ==================================================
        # ENSEMBLE
        # ==================================================

        previsoes_min = [

            modelo["previsao"]

            for modelo

            in oferta["modelos"]["min"].values()

            if modelo is not None

        ]

        previsoes_max = [

            modelo["previsao"]

            for modelo

            in oferta["modelos"]["max"].values()

            if modelo is not None

        ]

        def resumo(previsoes):

            if len(previsoes) == 0:

                return {

                    "n_modelos": 0,

                    "media": np.nan,

                    "mediana": np.nan,

                    "desvio": np.nan,

                    "minimo": np.nan,

                    "maximo": np.nan,

                    "amplitude": np.nan,

                }

            previsoes = np.asarray(previsoes)

            return {

                "n_modelos":
                    len(previsoes),

                "media":
                    float(np.mean(previsoes)),

                "mediana":
                    float(np.median(previsoes)),

                "desvio":
                    float(np.std(
                        previsoes,
                        ddof=1
                    )) if len(previsoes) > 1 else 0.0,

                "minimo":
                    float(np.min(previsoes)),

                "maximo":
                    float(np.max(previsoes)),

                "amplitude":
                    float(
                        np.max(previsoes)
                        - np.min(previsoes)
                    ) if len(previsoes) > 1 else 0.0,

            }

        oferta["ensemble"] = {

            "min": resumo(previsoes_min),

            "max": resumo(previsoes_max),

        }

    return dict_series


# ==========================================================
# EXECUÇÃO
# ==========================================================

dict_series = executar_modelos(dict_series)

# ==========================================================
# TESTE
# ==========================================================

exemplo = next(iter(dict_series.values()))

print("=" * 60)
print("EXECUTOR DOS MODELOS")
print("=" * 60)

print()

print("Oferta")

print(exemplo["curso"])
print(exemplo["campus"])
print(exemplo["turno"])
print(exemplo["modalidade"])

print()

resultado = []

for nome, modelo in exemplo["modelos"]["max"].items():

    if modelo is None:
        continue

    resultado.append({

        "Modelo":
            modelo["modelo"],

        "Previsão":
            modelo["previsao"],

        "R²":
            modelo["r2"],

        "Coeficiente":
            modelo["coef_angular"]

    })

display(pd.DataFrame(resultado))

print()

print("Ensemble")

display(pd.DataFrame([exemplo["ensemble"]["max"]]))

EXECUTOR DOS MODELOS

Oferta
Administração (Bacharelado)
Darcy Ribeiro
Diurno
AC



,Modelo,Previsão,R²,Coeficiente
0,Persistência,99.156,NaN,NaN
1,Regressão Linear,64.876,0.061,-5.006
2,Regressão Ponderada,62.418,0.059,-5.928
3,Média Móvel,78.226,NaN,NaN



Ensemble


,n_modelos,media,mediana,desvio,minimo,maximo,amplitude
0,4,76.169,71.551,16.825,62.418,99.156,36.738


In [202]:
# ==========================================================
# PARTE 3E.6A
# CONSOLIDAÇÃO DAS PREVISÕES
# DEFINIÇÃO DA FUNÇÃO
# ==========================================================

def consolidar_predicoes(dict_series):
    """
    Consolida toda a infraestrutura estatística construída
    na Parte 3E em um único DataFrame.

    O DataFrame resultante contém:

    • identificação da oferta;
    • histórico da série;
    • métricas descritivas;
    • previsões individuais;
    • métricas dos modelos;
    • estatísticas do ensemble.

    Nenhum cálculo estatístico é realizado nesta etapa.
    Apenas ocorre a consolidação das informações já
    armazenadas em dict_series.
    """

    registros = []

    for oferta in dict_series.values():

        registro = {

            # ==================================================
            # IDENTIFICAÇÃO
            # ==================================================

            "id_oferta":
                oferta["id_oferta"],

            "id_oferta_num":
                oferta["id_oferta_num"],

            "curso":
                oferta["curso"],

            "campus":
                oferta["campus"],

            "turno":
                oferta["turno"],

            "modalidade":
                oferta["modalidade"],

            # ==================================================
            # TAMANHO DAS SÉRIES
            # ==================================================

            "observacoes_min":
                len(oferta["y_min"]),

            "observacoes_max":
                len(oferta["y_max"]),

        }

        # ======================================================
        # PRIMEIRA E ÚLTIMA OBSERVAÇÃO
        # ======================================================

        for tipo in ["min", "max"]:

            y = oferta[f"y_{tipo}"]

            registro[f"primeira_nota_{tipo}"] = (

                float(y[0])

                if len(y) > 0

                else np.nan

            )

            registro[f"ultima_nota_{tipo}"] = (

                float(y[-1])

                if len(y) > 0

                else np.nan

            )

        # ======================================================
        # MÉTRICAS HISTÓRICAS
        # ======================================================

        for tipo in ["min", "max"]:

            met = oferta["metricas"][tipo]

            registro[f"media_historica_{tipo}"] = (
                met["media"]
            )

            registro[f"mediana_historica_{tipo}"] = (
                met["mediana"]
            )

            registro[f"minimo_historico_{tipo}"] = (
                met["minimo"]
            )

            registro[f"maximo_historico_{tipo}"] = (
                met["maximo"]
            )

            registro[f"amplitude_historica_{tipo}"] = (
                met["amplitude"]
            )

            registro[f"desvio_historico_{tipo}"] = (
                met["desvio"]
            )

            registro[f"coef_var_{tipo}"] = (
                met["coef_var"]
            )

            registro[f"consistencia_{tipo}"] = (
                met["consistencia"]
            )

            registro[f"mudancas_{tipo}"] = (
                met["mudancas"]
            )

            registro[f"volatilidade_{tipo}"] = (
                met["volatilidade"]
            )

        # ======================================================
        # MODELOS PREDITIVOS
        # ======================================================

        for tipo in ["min", "max"]:

            for nome in MODELOS.keys():

                modelo = oferta["modelos"][tipo][nome]

                if modelo is None:

                    registro[f"{nome}_{tipo}"] = np.nan
                    registro[f"r2_{nome}_{tipo}"] = np.nan

                else:

                    registro[f"{nome}_{tipo}"] = (
                        modelo["previsao"]
                    )

                    registro[f"r2_{nome}_{tipo}"] = (
                        modelo["r2"]
                    )

        # ======================================================
        # ENSEMBLE
        # ======================================================

        for tipo in ["min", "max"]:

            ens = oferta["ensemble"][tipo]

            registro[f"ensemble_{tipo}"] = (
                ens["media"]
            )

            registro[f"ensemble_mediana_{tipo}"] = (
                ens["mediana"]
            )

            registro[f"ensemble_desvio_{tipo}"] = (
                ens["desvio"]
            )

            registro[f"ensemble_amplitude_{tipo}"] = (
                ens["amplitude"]
            )

            registro[f"ensemble_n_modelos_{tipo}"] = (
                ens["n_modelos"]
            )

        registros.append(registro)

    predicoes = (

        pd.DataFrame(registros)

        .sort_values("id_oferta_num")

        .reset_index(drop=True)

        .round(3)

    )

    return predicoes

In [203]:
# ==========================================================
# PARTE 3E.6B
# EXECUÇÃO DA CONSOLIDAÇÃO
# ==========================================================

# ==========================================================
# EXECUÇÃO
# ==========================================================

predicoes = consolidar_predicoes(dict_series)

# ==========================================================
# RELATÓRIO GERAL
# ==========================================================

print("=" * 80)
print("CONSOLIDAÇÃO DAS PREVISÕES")
print("=" * 80)

print(f"Total de ofertas.............. {len(predicoes):,}")
print(f"Total de colunas.............. {predicoes.shape[1]}")

print()

print("Primeiras colunas")

for i, coluna in enumerate(predicoes.columns, start=1):

    print(f"{i:02d} - {coluna}")

print()

print("=" * 80)
print("ESTATÍSTICAS DO ENSEMBLE")
print("=" * 80)

display(

    predicoes[

        [

            "ensemble_min",

            "ensemble_max",

            "ensemble_desvio_min",

            "ensemble_desvio_max",

            "ensemble_amplitude_min",

            "ensemble_amplitude_max",

            "ensemble_n_modelos_min",

            "ensemble_n_modelos_max",

        ]

    ]

    .describe()

    .round(3)

)

print()

print("=" * 80)
print("ESTATÍSTICAS HISTÓRICAS")
print("=" * 80)

display(

    predicoes[

        [

            "media_historica_min",

            "media_historica_max",

            "desvio_historico_min",

            "desvio_historico_max",

            "coef_var_min",

            "coef_var_max",

            "consistencia_min",

            "consistencia_max",

            "volatilidade_min",

            "volatilidade_max",

        ]

    ]

    .describe()

    .round(3)

)

print()

print("=" * 80)
print("AMOSTRA DA BASE CONSOLIDADA")
print("=" * 80)

display(predicoes.head())

# ==========================================================
# EXEMPLO
# ==========================================================

exemplo = predicoes.iloc[0]

print("=" * 80)
print("EXEMPLO DE OFERTA")
print("=" * 80)

print(f"Curso......................... {exemplo['curso']}")
print(f"Campus........................ {exemplo['campus']}")
print(f"Turno......................... {exemplo['turno']}")
print(f"Modalidade.................... {exemplo['modalidade']}")

print()

print("Histórico")

print(f"Observações (mín.)............ {int(exemplo['observacoes_min'])}")
print(f"Observações (máx.)............ {int(exemplo['observacoes_max'])}")

print(f"Primeira nota (mín.).......... {exemplo['primeira_nota_min']:.2f}")
print(f"Última nota (mín.)............ {exemplo['ultima_nota_min']:.2f}")

print(f"Primeira nota (máx.).......... {exemplo['primeira_nota_max']:.2f}")
print(f"Última nota (máx.)............ {exemplo['ultima_nota_max']:.2f}")

print()

print("Previsões")

print(f"Ensemble mínimo............... {exemplo['ensemble_min']:.2f}")
print(f"Ensemble máximo............... {exemplo['ensemble_max']:.2f}")

print()

print("Dispersão")

print(f"Modelos (mín.)................ {int(exemplo['ensemble_n_modelos_min'])}")
print(f"Modelos (máx.)................ {int(exemplo['ensemble_n_modelos_max'])}")

print(f"Desvio (mín.)................. {exemplo['ensemble_desvio_min']:.2f}")
print(f"Desvio (máx.)................. {exemplo['ensemble_desvio_max']:.2f}")

print(f"Amplitude (mín.).............. {exemplo['ensemble_amplitude_min']:.2f}")
print(f"Amplitude (máx.).............. {exemplo['ensemble_amplitude_max']:.2f}")

print()

print("Variabilidade histórica")

print(f"Desvio histórico (mín.)....... {exemplo['desvio_historico_min']:.2f}")
print(f"Desvio histórico (máx.)....... {exemplo['desvio_historico_max']:.2f}")

print(f"Coeficiente de variação (mín.) {exemplo['coef_var_min']:.2f}%")
print(f"Coeficiente de variação (máx.) {exemplo['coef_var_max']:.2f}%")

print()

print("Consolidação concluída.")

CONSOLIDAÇÃO DAS PREVISÕES
Total de ofertas.............. 765
Total de colunas.............. 58

Primeiras colunas
01 - id_oferta
02 - id_oferta_num
03 - curso
04 - campus
05 - turno
06 - modalidade
07 - observacoes_min
08 - observacoes_max
09 - primeira_nota_min
10 - ultima_nota_min
11 - primeira_nota_max
12 - ultima_nota_max
13 - media_historica_min
14 - mediana_historica_min
15 - minimo_historico_min
16 - maximo_historico_min
17 - amplitude_historica_min
18 - desvio_historico_min
19 - coef_var_min
20 - consistencia_min
21 - mudancas_min
22 - volatilidade_min
23 - media_historica_max
24 - mediana_historica_max
25 - minimo_historico_max
26 - maximo_historico_max
27 - amplitude_historica_max
28 - desvio_historico_max
29 - coef_var_max
30 - consistencia_max
31 - mudancas_max
32 - volatilidade_max
33 - persistencia_min
34 - r2_persistencia_min
35 - regressao_linear_min
36 - r2_regressao_linear_min
37 - regressao_ponderada_min
38 - r2_regressao_ponderada_min
39 - media_movel_min
40 - r2_m

,ensemble_min,ensemble_max,ensemble_desvio_min,ensemble_desvio_max,ensemble_amplitude_min,ensemble_amplitude_max,ensemble_n_modelos_min,ensemble_n_modelos_max
count,764.000,765.000,764.000,765.000,764.000,765.000,765.000,765.000
mean,-17.853,8.629,8.328,9.397,18.090,20.380,3.292,3.294
std,45.305,47.071,9.312,10.027,19.926,21.533,0.963,0.956
min,-106.206,-97.360,0.000,0.000,0.000,0.000,0.000,2.000
25%,-49.629,-23.818,0.000,0.000,0.000,0.000,2.000,2.000
50%,-27.007,1.702,6.424,7.611,14.292,16.433,4.000,4.000
75%,4.734,32.559,13.122,15.214,29.326,33.416,4.000,4.000
max,167.330,188.249,58.874,58.874,126.346,122.914,4.000,4.000



ESTATÍSTICAS HISTÓRICAS


,media_historica_min,media_historica_max,desvio_historico_min,desvio_historico_max,coef_var_min,coef_var_max,consistencia_min,consistencia_max,volatilidade_min,volatilidade_max
count,764.000,765.000,764.000,765.000,764.000,765.000,764.000,765.000,764.000,765.000
mean,-18.470,11.100,16.737,18.134,-28.884,-44.349,81.381,80.209,23.019,25.398
std,40.876,43.931,16.830,17.552,373.754,2396.896,20.131,20.585,27.705,29.611
min,-91.423,-90.203,0.000,0.000,-4344.623,-38623.405,50.000,50.000,0.000,0.000
25%,-45.556,-19.449,0.000,0.000,-69.468,-21.400,66.667,60.000,0.000,0.000
50%,-26.995,3.254,14.797,16.551,0.000,0.000,100.000,75.000,14.685,16.335
75%,-2.402,33.304,28.080,30.127,0.000,60.051,100.000,100.000,38.753,44.524
max,171.083,194.988,83.630,77.316,3541.599,27530.354,100.000,100.000,147.284,162.643



AMOSTRA DA BASE CONSOLIDADA


,id_oferta,id_oferta_num,curso,campus,turno,modalidade,observacoes_min,observacoes_max,primeira_nota_min,ultima_nota_min,primeira_nota_max,ultima_nota_max,media_historica_min,mediana_historica_min,minimo_historico_min,maximo_historico_min,amplitude_historica_min,desvio_historico_min,coef_var_min,consistencia_min,mudancas_min,volatilidade_min,media_historica_max,mediana_historica_max,minimo_historico_max,maximo_historico_max,amplitude_historica_max,desvio_historico_max,coef_var_max,consistencia_max,mudancas_max,volatilidade_max,persistencia_min,r2_persistencia_min,regressao_linear_min,r2_regressao_linear_min,regressao_ponderada_min,r2_regressao_ponderada_min,media_movel_min,r2_media_movel_min,persistencia_max,r2_persistencia_max,regressao_linear_max,r2_regressao_linear_max,regressao_ponderada_max,r2_regressao_ponderada_max,media_movel_max,r2_media_movel_max,ensemble_min,ensemble_mediana_min,ensemble_desvio_min,ensemble_amplitude_min,ensemble_n_modelos_min,ensemble_max,ensemble_mediana_max,ensemble_desvio_max,ensemble_amplitude_max,ensemble_n_modelos_max
0,Administração (Bacharelado) | Darcy Ribeiro | ...,1,Administração (Bacharelado),Darcy Ribeiro,Diurno,AC,6,6,-5.813,38.131,69.035,99.156,26.868,37.418,-5.813,46.485,52.298,23.412,87.136,80.000,2.000,35.681,82.398,96.546,12.807,120.300,107.493,37.802,45.877,60.000,2,65.453,38.131,NaN,37.962,0.064,28.688,-0.031,29.510,NaN,99.156,NaN,64.876,0.061,62.418,0.059,78.226,NaN,33.573,33.736,5.177,9.443,4,76.169,71.551,16.825,36.738,4
1,Administração (Bacharelado) | Darcy Ribeiro | ...,2,Administração (Bacharelado),Darcy Ribeiro,Noturno,AC,6,6,-98.900,-7.050,38.986,44.901,-38.006,-16.271,-98.900,-2.777,96.123,42.970,-113.060,60.000,2.000,68.732,40.601,47.522,-40.646,98.221,138.867,45.085,111.043,60.000,4,73.686,-7.050,NaN,-17.479,0.065,-29.290,0.019,-33.119,NaN,44.901,NaN,2.085,0.209,0.305,0.208,31.431,NaN,-21.734,-23.384,11.838,26.069,4,19.680,16.758,22.054,44.596,4
2,Administração (Bacharelado) | Darcy Ribeiro | ...,3,Administração (Bacharelado),Darcy Ribeiro,Diurno,CN,6,6,-40.808,-12.580,3.022,6.842,-28.120,-26.694,-54.707,-4.981,49.726,22.874,-81.344,60.000,4.000,48.142,-7.373,1.368,-49.856,44.699,94.555,36.280,-492.069,60.000,3,76.162,-12.580,NaN,-24.194,0.008,-23.703,0.008,-27.185,NaN,6.842,NaN,-10.520,0.002,-5.991,-0.007,-8.122,NaN,-21.916,-23.949,6.411,14.605,4,-4.448,-7.057,7.751,17.362,4
3,Administração (Bacharelado) | Darcy Ribeiro | ...,4,Administração (Bacharelado),Darcy Ribeiro,Noturno,CN,5,5,-3.806,-54.846,1.212,-53.830,-23.060,-6.548,-54.846,-1.176,53.670,26.464,-114.765,50.000,3.000,43.831,-6.654,1.212,-53.830,19.031,72.861,27.610,-414.944,50.000,1,29.730,-54.846,NaN,-58.810,0.468,-59.141,0.467,-30.223,NaN,-53.830,NaN,-48.142,0.579,-59.904,0.455,-14.281,NaN,-50.755,-56.828,13.826,28.918,4,-44.039,-50.986,20.412,45.623,4
4,Administração (Bacharelado) | Darcy Ribeiro | ...,5,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,4,4,-90.624,-59.221,-3.524,-21.197,-51.749,-50.495,-90.624,-15.383,75.241,31.566,-60.997,66.667,1.000,56.273,-2.144,-1.766,-21.197,16.153,37.350,15.322,-714.546,66.667,1,27.969,-59.221,NaN,-34.793,0.077,-53.332,-0.096,-48.358,NaN,-21.197,NaN,-11.358,0.096,-19.532,-0.046,-3.987,NaN,-48.926,-50.845,10.416,24.428,4,-14.019,-15.445,7.950,17.210,4


EXEMPLO DE OFERTA
Curso......................... Administração (Bacharelado)
Campus........................ Darcy Ribeiro
Turno......................... Diurno
Modalidade.................... AC

Histórico
Observações (mín.)............ 6
Observações (máx.)............ 6
Primeira nota (mín.).......... -5.81
Última nota (mín.)............ 38.13
Primeira nota (máx.).......... 69.03
Última nota (máx.)............ 99.16

Previsões
Ensemble mínimo............... 33.57
Ensemble máximo............... 76.17

Dispersão
Modelos (mín.)................ 4
Modelos (máx.)................ 4
Desvio (mín.)................. 5.18
Desvio (máx.)................. 16.82
Amplitude (mín.).............. 9.44
Amplitude (máx.).............. 36.74

Variabilidade histórica
Desvio histórico (mín.)....... 23.41
Desvio histórico (máx.)....... 37.80
Coeficiente de variação (mín.) 87.14%
Coeficiente de variação (máx.) 45.88%

Consolidação concluída.


In [204]:
# ==========================================================
# PARTE 4.1 — AUDITORIA DA BASE DE PREDIÇÕES
# ==========================================================
#
# Objetivo
# --------
# Validar a cobertura da infraestrutura preditiva construída
# na Parte 3E.
#
# Esta etapa NÃO recalcula modelos.
# Apenas inspeciona o DataFrame "predicoes".
#
# São avaliadas separadamente as previsões das notas mínimas
# e máximas.
# ==========================================================

print("=" * 80)
print("AUDITORIA DA BASE DE PREDIÇÕES")
print("=" * 80)

total = len(predicoes)

print(f"\nTotal de ofertas: {total:,}")

# ==========================================================
# Cobertura - Nota mínima
# ==========================================================

modelos_min = [
    "persistencia_min",
    "regressao_linear_min",
    "regressao_ponderada_min",
    "media_movel_min",
    "ensemble_min"
]

resumo_min = []

for coluna in modelos_min:

    resumo_min.append({
        "Modelo": coluna.replace("_min", ""),
        "Previsões": predicoes[coluna].notna().sum(),
        "Cobertura (%)": round(
            predicoes[coluna].notna().mean() * 100,
            2
        )
    })

df_min = pd.DataFrame(resumo_min)

print("\n")
print("=" * 80)
print("COBERTURA DOS MODELOS — NOTA MÍNIMA")
print("=" * 80)

display(df_min)

# ==========================================================
# Cobertura - Nota máxima
# ==========================================================

modelos_max = [
    "persistencia_max",
    "regressao_linear_max",
    "regressao_ponderada_max",
    "media_movel_max",
    "ensemble_max"
]

resumo_max = []

for coluna in modelos_max:

    resumo_max.append({
        "Modelo": coluna.replace("_max", ""),
        "Previsões": predicoes[coluna].notna().sum(),
        "Cobertura (%)": round(
            predicoes[coluna].notna().mean() * 100,
            2
        )
    })

df_max = pd.DataFrame(resumo_max)

print("\n")
print("=" * 80)
print("COBERTURA DOS MODELOS — NOTA MÁXIMA")
print("=" * 80)

display(df_max)

# ==========================================================
# Distribuição do tamanho das séries
# ==========================================================

print("\n")
print("=" * 80)
print("DISTRIBUIÇÃO DAS OBSERVAÇÕES HISTÓRICAS")
print("=" * 80)

obs = (
    predicoes["observacoes_min"]
    .fillna(0)
    .astype(int)
    .value_counts()
    .sort_index()
    .rename_axis("Nº de observações")
    .reset_index(name="Ofertas")
)

obs["Percentual (%)"] = (
    obs["Ofertas"] / total * 100
).round(2)

display(obs)

# ==========================================================
# Estatísticas do ensemble
# ==========================================================

print("\n")
print("=" * 80)
print("ENSEMBLE")
print("=" * 80)

print(
    f"Ensemble (mínimo) disponível : "
    f"{predicoes['ensemble_min'].notna().sum():,}"
)

print(
    f"Ensemble (máximo) disponível : "
    f"{predicoes['ensemble_max'].notna().sum():,}"
)

print(
    f"Nº médio de modelos (mínimo): "
    f"{predicoes['ensemble_n_modelos_min'].mean():.2f}"
)

print(
    f"Nº médio de modelos (máximo): "
    f"{predicoes['ensemble_n_modelos_max'].mean():.2f}"
)

print("\nAuditoria concluída.")

# ==========================================================
# Auditoria das métricas históricas
# ==========================================================

print("\n")
print("=" * 80)
print("MÉTRICAS HISTÓRICAS")
print("=" * 80)

metricas = [
    "media_historica_min",
    "desvio_historico_min",
    "coef_var_min",
    "consistencia_min",
    "volatilidade_min",
]

auditoria_metricas = []

for coluna in metricas:

    auditoria_metricas.append({

        "Métrica": coluna,

        "Valores válidos": predicoes[coluna].notna().sum(),

        "Cobertura (%)": round(
            predicoes[coluna].notna().mean() * 100,
            2
        )

    })

display(pd.DataFrame(auditoria_metricas))

AUDITORIA DA BASE DE PREDIÇÕES

Total de ofertas: 765


COBERTURA DOS MODELOS — NOTA MÍNIMA


,Modelo,Previsões,Cobertura (%)
0,persistencia,764,99.870
1,regressao_linear,495,64.710
2,regressao_ponderada,495,64.710
3,media_movel,764,99.870
4,ensemble,764,99.870




COBERTURA DOS MODELOS — NOTA MÁXIMA


,Modelo,Previsões,Cobertura (%)
0,persistencia,765,100.000
1,regressao_linear,495,64.710
2,regressao_ponderada,495,64.710
3,media_movel,765,100.000
4,ensemble,765,100.000




DISTRIBUIÇÃO DAS OBSERVAÇÕES HISTÓRICAS


,Nº de observações,Ofertas,Percentual (%)
0,0,1,0.130
1,1,269,35.160
2,2,63,8.240
3,3,94,12.290
4,4,139,18.170
5,5,126,16.470
6,6,73,9.540




ENSEMBLE
Ensemble (mínimo) disponível : 764
Ensemble (máximo) disponível : 765
Nº médio de modelos (mínimo): 3.29
Nº médio de modelos (máximo): 3.29

Auditoria concluída.


MÉTRICAS HISTÓRICAS


,Métrica,Valores válidos,Cobertura (%)
0,media_historica_min,764,99.870
1,desvio_historico_min,764,99.870
2,coef_var_min,764,99.870
3,consistencia_min,764,99.870
4,volatilidade_min,764,99.870


In [205]:
# ==========================================================
# PARTE 4.2 — CARACTERIZAÇÃO DAS SÉRIES HISTÓRICAS
# ==========================================================
#
# Objetivo
# --------
# Caracterizar o comportamento histórico das notas mínimas
# e máximas a partir das métricas calculadas na Parte 3E.
#
# Esta etapa NÃO recalcula nenhuma métrica.
# Utiliza exclusivamente o DataFrame "predicoes".
# ==========================================================

print("=" * 80)
print("CARACTERIZAÇÃO DAS SÉRIES HISTÓRICAS")
print("=" * 80)

# ==========================================================
# Estatísticas descritivas — Nota mínima
# ==========================================================

print("\n")
print("=" * 80)
print("NOTAS MÍNIMAS")
print("=" * 80)

estatisticas_min = (
    predicoes[
        [
            "media_historica_min",
            "mediana_historica_min",
            "minimo_historico_min",
            "maximo_historico_min",
            "amplitude_historica_min",
            "desvio_historico_min",
            "coef_var_min",
            "consistencia_min",
            "mudancas_min",
            "volatilidade_min",
        ]
    ]
    .describe()
    .T
    .round(3)
)

display(estatisticas_min)

# ==========================================================
# Estatísticas descritivas — Nota máxima
# ==========================================================

print("\n")
print("=" * 80)
print("NOTAS MÁXIMAS")
print("=" * 80)

estatisticas_max = (
    predicoes[
        [
            "media_historica_max",
            "mediana_historica_max",
            "minimo_historico_max",
            "maximo_historico_max",
            "amplitude_historica_max",
            "desvio_historico_max",
            "coef_var_max",
            "consistencia_max",
            "mudancas_max",
            "volatilidade_max",
        ]
    ]
    .describe()
    .T
    .round(3)
)

display(estatisticas_max)

# ==========================================================
# Resumo consolidado
# ==========================================================

print("\n")
print("=" * 80)
print("RESUMO CONSOLIDADO")
print("=" * 80)

resumo = pd.DataFrame({

    "Indicador": [

        "Amplitude histórica média",
        "Desvio-padrão médio",
        "Coeficiente de variação médio (%)",
        "Consistência média (%)",
        "Mudanças médias de tendência",
        "Volatilidade média"

    ],

    "Nota mínima": [

        predicoes["amplitude_historica_min"].mean(),
        predicoes["desvio_historico_min"].mean(),
        predicoes["coef_var_min"].mean(),
        predicoes["consistencia_min"].mean(),
        predicoes["mudancas_min"].mean(),
        predicoes["volatilidade_min"].mean()

    ],

    "Nota máxima": [

        predicoes["amplitude_historica_max"].mean(),
        predicoes["desvio_historico_max"].mean(),
        predicoes["coef_var_max"].mean(),
        predicoes["consistencia_max"].mean(),
        predicoes["mudancas_max"].mean(),
        predicoes["volatilidade_max"].mean()

    ]

}).round(3)

display(resumo)

print("\nCaracterização concluída.")

CARACTERIZAÇÃO DAS SÉRIES HISTÓRICAS


NOTAS MÍNIMAS


,count,mean,std,min,25%,50%,75%,max
media_historica_min,764.000,-18.470,40.876,-91.423,-45.556,-26.995,-2.402,171.083
mediana_historica_min,764.000,-17.303,42.274,-92.488,-46.712,-25.572,2.240,166.750
minimo_historico_min,764.000,-37.802,42.027,-105.487,-65.596,-45.976,-20.842,162.424
maximo_historico_min,764.000,-0.970,47.254,-91.423,-34.193,-10.320,25.338,183.500
amplitude_historica_min,764.000,36.832,38.040,0.000,0.000,30.234,62.292,195.713
desvio_historico_min,764.000,16.737,16.830,0.000,0.000,14.797,28.080,83.630
coef_var_min,764.000,-28.884,373.754,-4344.623,-69.468,0.000,0.000,3541.599
consistencia_min,764.000,81.381,20.131,50.000,66.667,100.000,100.000,100.000
mudancas_min,764.000,0.869,1.079,0.000,0.000,0.000,2.000,4.000
volatilidade_min,764.000,23.019,27.705,0.000,0.000,14.685,38.753,147.284




NOTAS MÁXIMAS


,count,mean,std,min,25%,50%,75%,max
media_historica_max,765.000,11.100,43.931,-90.203,-19.449,3.254,33.304,194.988
mediana_historica_max,765.000,12.376,45.440,-90.203,-19.577,3.966,35.100,190.742
minimo_historico_max,765.000,-10.670,41.933,-105.487,-39.988,-16.980,11.211,173.057
maximo_historico_max,765.000,29.938,52.748,-90.203,-7.430,22.756,59.008,211.738
amplitude_historica_max,765.000,40.608,40.922,0.000,0.000,33.828,68.766,198.588
desvio_historico_max,765.000,18.134,17.552,0.000,0.000,16.551,30.127,77.316
coef_var_max,765.000,-44.349,2396.896,-38623.405,-21.400,0.000,60.051,27530.354
consistencia_max,765.000,80.209,20.585,50.000,60.000,75.000,100.000,100.000
mudancas_max,765.000,0.903,1.054,0.000,0.000,1.000,2.000,4.000
volatilidade_max,765.000,25.398,29.611,0.000,0.000,16.335,44.524,162.643




RESUMO CONSOLIDADO


,Indicador,Nota mínima,Nota máxima
0,Amplitude histórica média,36.832,40.608
1,Desvio-padrão médio,16.737,18.134
2,Coeficiente de variação médio (%),-28.884,-44.349
3,Consistência média (%),81.381,80.209
4,Mudanças médias de tendência,0.869,0.903
5,Volatilidade média,23.019,25.398



Caracterização concluída.


In [206]:
# ==========================================================
# PARTE 4.3 — COBERTURA DOS MODELOS PREDITIVOS
# ==========================================================
#
# Objetivo
# --------
# Avaliar a utilização efetiva dos modelos preditivos
# produzidos pela Parte 3E.
#
# Esta etapa identifica:
#
# • cobertura individual de cada modelo;
# • quantidade média de modelos por oferta;
# • distribuição do número de modelos utilizados
#   na composição do ensemble.
#
# Nenhum modelo é recalculado.
# ==========================================================

print("=" * 80)
print("COBERTURA DOS MODELOS PREDITIVOS")
print("=" * 80)

total = len(predicoes)

# ==========================================================
# Cobertura - Nota mínima
# ==========================================================

print("\n")
print("=" * 80)
print("COBERTURA — NOTA MÍNIMA")
print("=" * 80)

modelos_min = [
    "persistencia_min",
    "regressao_linear_min",
    "regressao_ponderada_min",
    "media_movel_min",
    "ensemble_min",
]

cobertura_min = []

for coluna in modelos_min:

    cobertura_min.append({

        "Modelo":
            coluna.replace("_min", "").replace("_", " ").title(),

        "Previsões":
            predicoes[coluna].notna().sum(),

        "Cobertura (%)":
            round(
                predicoes[coluna].notna().mean() * 100,
                2
            )

    })

display(pd.DataFrame(cobertura_min))

# ==========================================================
# Cobertura - Nota máxima
# ==========================================================

print("\n")
print("=" * 80)
print("COBERTURA — NOTA MÁXIMA")
print("=" * 80)

modelos_max = [
    "persistencia_max",
    "regressao_linear_max",
    "regressao_ponderada_max",
    "media_movel_max",
    "ensemble_max",
]

cobertura_max = []

for coluna in modelos_max:

    cobertura_max.append({

        "Modelo":
            coluna.replace("_max", "").replace("_", " ").title(),

        "Previsões":
            predicoes[coluna].notna().sum(),

        "Cobertura (%)":
            round(
                predicoes[coluna].notna().mean() * 100,
                2
            )

    })

display(pd.DataFrame(cobertura_max))

# ==========================================================
# Distribuição do número de modelos
# ==========================================================

print("\n")
print("=" * 80)
print("NÚMERO DE MODELOS UTILIZADOS")
print("=" * 80)

dist_modelos = (
    predicoes[
        [
            "ensemble_n_modelos_min",
            "ensemble_n_modelos_max"
        ]
    ]
    .rename(columns={
        "ensemble_n_modelos_min": "Nota mínima",
        "ensemble_n_modelos_max": "Nota máxima"
    })
)

distribuicao = pd.DataFrame({

    "Modelos":

        sorted(

            set(dist_modelos["Nota mínima"].dropna().astype(int))

            |

            set(dist_modelos["Nota máxima"].dropna().astype(int))

        )

})

distribuicao["Ofertas (mín.)"] = distribuicao["Modelos"].apply(

    lambda x:

    (predicoes["ensemble_n_modelos_min"] == x).sum()

)

distribuicao["Ofertas (máx.)"] = distribuicao["Modelos"].apply(

    lambda x:

    (predicoes["ensemble_n_modelos_max"] == x).sum()

)

distribuicao["Percentual (mín.)"] = (

    distribuicao["Ofertas (mín.)"]

    / total

    * 100

).round(2)

distribuicao["Percentual (máx.)"] = (

    distribuicao["Ofertas (máx.)"]

    / total

    * 100

).round(2)

display(distribuicao)

# ==========================================================
# Estatísticas gerais
# ==========================================================

print("\n")
print("=" * 80)
print("ESTATÍSTICAS GERAIS")
print("=" * 80)

estatisticas = pd.DataFrame({

    "Indicador": [

        "Número médio de modelos",
        "Número mediano de modelos",
        "Número mínimo de modelos",
        "Número máximo de modelos",

    ],

    "Nota mínima": [

        predicoes["ensemble_n_modelos_min"].mean(),
        predicoes["ensemble_n_modelos_min"].median(),
        predicoes["ensemble_n_modelos_min"].min(),
        predicoes["ensemble_n_modelos_min"].max(),

    ],

    "Nota máxima": [

        predicoes["ensemble_n_modelos_max"].mean(),
        predicoes["ensemble_n_modelos_max"].median(),
        predicoes["ensemble_n_modelos_max"].min(),
        predicoes["ensemble_n_modelos_max"].max(),

    ]

}).round(3)

display(estatisticas)

print("\nCobertura dos modelos concluída.")

COBERTURA DOS MODELOS PREDITIVOS


COBERTURA — NOTA MÍNIMA


,Modelo,Previsões,Cobertura (%)
0,Persistencia,764,99.870
1,Regressao Linear,495,64.710
2,Regressao Ponderada,495,64.710
3,Media Movel,764,99.870
4,Ensemble,764,99.870




COBERTURA — NOTA MÁXIMA


,Modelo,Previsões,Cobertura (%)
0,Persistencia,765,100.000
1,Regressao Linear,495,64.710
2,Regressao Ponderada,495,64.710
3,Media Movel,765,100.000
4,Ensemble,765,100.000




NÚMERO DE MODELOS UTILIZADOS


,Modelos,Ofertas (mín.),Ofertas (máx.),Percentual (mín.),Percentual (máx.)
0,0,1,0,0.130,0.000
1,2,269,270,35.160,35.290
2,4,495,495,64.710,64.710




ESTATÍSTICAS GERAIS


,Indicador,Nota mínima,Nota máxima
0,Número médio de modelos,3.292,3.294
1,Número mediano de modelos,4.000,4.000
2,Número mínimo de modelos,0.000,2.000
3,Número máximo de modelos,4.000,4.000



Cobertura dos modelos concluída.


In [207]:
# ==========================================================
# PARTE 4.4 — ANÁLISE DO ENSEMBLE
# ==========================================================
#
# Objetivo
# --------
# Caracterizar o comportamento do ensemble construído na
# Parte 3E.
#
# São analisados:
#
# • previsão média do ensemble;
# • dispersão entre os modelos;
# • amplitude entre as previsões;
# • quantidade de modelos utilizados.
#
# Nenhum modelo é recalculado.
# ==========================================================

print("=" * 80)
print("ANÁLISE DO ENSEMBLE")
print("=" * 80)

# ==========================================================
# Estatísticas — Nota mínima
# ==========================================================

print("\n")
print("=" * 80)
print("ENSEMBLE — NOTA MÍNIMA")
print("=" * 80)

ensemble_min = (
    predicoes[
        [
            "ensemble_min",
            "ensemble_mediana_min",
            "ensemble_desvio_min",
            "ensemble_amplitude_min",
            "ensemble_n_modelos_min",
        ]
    ]
    .describe()
    .T
    .round(3)
)

display(ensemble_min)

# ==========================================================
# Estatísticas — Nota máxima
# ==========================================================

print("\n")
print("=" * 80)
print("ENSEMBLE — NOTA MÁXIMA")
print("=" * 80)

ensemble_max = (
    predicoes[
        [
            "ensemble_max",
            "ensemble_mediana_max",
            "ensemble_desvio_max",
            "ensemble_amplitude_max",
            "ensemble_n_modelos_max",
        ]
    ]
    .describe()
    .T
    .round(3)
)

display(ensemble_max)

# ==========================================================
# Resumo comparativo
# ==========================================================

print("\n")
print("=" * 80)
print("RESUMO COMPARATIVO")
print("=" * 80)

comparativo = pd.DataFrame({

    "Indicador": [

        "Previsão média",
        "Previsão mediana",
        "Desvio médio do ensemble",
        "Amplitude média do ensemble",
        "Número médio de modelos"

    ],

    "Nota mínima": [

        predicoes["ensemble_min"].mean(),
        predicoes["ensemble_mediana_min"].mean(),
        predicoes["ensemble_desvio_min"].mean(),
        predicoes["ensemble_amplitude_min"].mean(),
        predicoes["ensemble_n_modelos_min"].mean(),

    ],

    "Nota máxima": [

        predicoes["ensemble_max"].mean(),
        predicoes["ensemble_mediana_max"].mean(),
        predicoes["ensemble_desvio_max"].mean(),
        predicoes["ensemble_amplitude_max"].mean(),
        predicoes["ensemble_n_modelos_max"].mean(),

    ]

}).round(3)

display(comparativo)

# ==========================================================
# Ofertas com maior dispersão entre os modelos
# ==========================================================

print("\n")
print("=" * 80)
print("MAIOR DISPERSÃO ENTRE OS MODELOS")
print("=" * 80)

colunas = [

    "curso",
    "campus",
    "turno",
    "modalidade",

    "ensemble_amplitude_min",
    "ensemble_amplitude_max",

    "ensemble_desvio_min",
    "ensemble_desvio_max",

    "ensemble_n_modelos_min",
    "ensemble_n_modelos_max",

]

maior_disp = (

    predicoes[colunas]

    .sort_values(

        by=[

            "ensemble_amplitude_min",
            "ensemble_amplitude_max"

        ],

        ascending=False

    )

    .head(15)

    .reset_index(drop=True)

)

display(maior_disp)

print("\nAnálise do ensemble concluída.")

ANÁLISE DO ENSEMBLE


ENSEMBLE — NOTA MÍNIMA


,count,mean,std,min,25%,50%,75%,max
ensemble_min,764.000,-17.853,45.305,-106.206,-49.629,-27.007,4.734,167.330
ensemble_mediana_min,764.000,-17.949,45.937,-106.808,-50.245,-26.370,4.722,167.719
ensemble_desvio_min,764.000,8.328,9.312,0.000,0.000,6.424,13.122,58.874
ensemble_amplitude_min,764.000,18.090,19.926,0.000,0.000,14.292,29.326,126.346
ensemble_n_modelos_min,765.000,3.292,0.963,0.000,2.000,4.000,4.000,4.000




ENSEMBLE — NOTA MÁXIMA


,count,mean,std,min,25%,50%,75%,max
ensemble_max,765.000,8.629,47.071,-97.360,-23.818,1.702,32.559,188.249
ensemble_mediana_max,765.000,8.032,47.647,-98.855,-24.926,0.728,32.373,189.139
ensemble_desvio_max,765.000,9.397,10.027,0.000,0.000,7.611,15.214,58.874
ensemble_amplitude_max,765.000,20.380,21.533,0.000,0.000,16.433,33.416,122.914
ensemble_n_modelos_max,765.000,3.294,0.956,2.000,2.000,4.000,4.000,4.000




RESUMO COMPARATIVO


,Indicador,Nota mínima,Nota máxima
0,Previsão média,-17.853,8.629
1,Previsão mediana,-17.949,8.032
2,Desvio médio do ensemble,8.328,9.397
3,Amplitude média do ensemble,18.090,20.380
4,Número médio de modelos,3.292,3.294




MAIOR DISPERSÃO ENTRE OS MODELOS


,curso,campus,turno,modalidade,ensemble_amplitude_min,ensemble_amplitude_max,ensemble_desvio_min,ensemble_desvio_max,ensemble_n_modelos_min,ensemble_n_modelos_max
0,Engenharia Ambiental (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_PPI,126.346,86.129,54.787,40.459,4,4
1,Física (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_PPI,122.914,122.914,58.874,58.874,4,4
2,Línguas Estrangeiras Aplicadas (Bacharelado),Darcy Ribeiro,Diurno,CN,105.405,105.405,54.325,54.325,4,4
3,Filosofia (Bacharelado/Licenciatura),Darcy Ribeiro,Diurno,EP_R2_NPPI,94.502,56.001,45.394,25.814,4,4
4,Saúde Coletiva (Bacharelado),Darcy Ribeiro,Noturno,EP_R2_PPI,90.877,33.472,46.837,17.251,4,4
5,Engenharia de Computação (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_NPPI,84.306,79.038,34.651,33.587,4,4
6,Gestão Ambiental (Bacharelado),Planaltina,Noturno,EP_R2_NPPI,83.708,66.461,43.142,34.253,4,4
7,Arquivologia (Bacharelado),Darcy Ribeiro,Noturno,CN,82.026,82.026,36.062,36.062,4,4
8,Matemática (Licenciatura),Darcy Ribeiro,Noturno,EP_R1_PPI,81.711,58.367,42.113,30.081,4,4
9,Letras – Português do Brasil como Segunda Líng...,Darcy Ribeiro,Diurno,EP_R2_PPI,79.893,50.344,36.859,23.089,4,4



Análise do ensemble concluída.


In [208]:
# ==========================================================
# PARTE 4.5 — CONSTRUÇÃO DA BASE ANALÍTICA
# ==========================================================
#
# Objetivo
# --------
# Construir uma base consolidada contendo as informações
# produzidas pela infraestrutura estatística da Parte 3E.
#
# Esta etapa não realiza cálculos.
# Apenas seleciona, organiza e prepara as variáveis para as
# análises subsequentes.
# ==========================================================

print("=" * 80)
print("CONSTRUÇÃO DA BASE ANALÍTICA")
print("=" * 80)

# ==========================================================
# Identificação
# ==========================================================

identificacao = [

    "id_oferta",
    "id_oferta_num",

    "curso",
    "campus",
    "turno",
    "modalidade",

]

# ==========================================================
# Histórico
# ==========================================================

historico = [

    "observacoes_min",
    "observacoes_max",

    "primeira_nota_min",
    "ultima_nota_min",

    "primeira_nota_max",
    "ultima_nota_max",

]

# ==========================================================
# Métricas históricas
# ==========================================================

metricas = [

    # ----------------------------
    # Nota mínima
    # ----------------------------

    "media_historica_min",
    "mediana_historica_min",

    "minimo_historico_min",
    "maximo_historico_min",

    "amplitude_historica_min",

    "desvio_historico_min",

    "coef_var_min",

    "consistencia_min",

    "mudancas_min",

    "volatilidade_min",

    # ----------------------------
    # Nota máxima
    # ----------------------------

    "media_historica_max",
    "mediana_historica_max",

    "minimo_historico_max",
    "maximo_historico_max",

    "amplitude_historica_max",

    "desvio_historico_max",

    "coef_var_max",

    "consistencia_max",

    "mudancas_max",

    "volatilidade_max",

]

# ==========================================================
# Modelos
# ==========================================================

modelos = [

    "persistencia_min",
    "persistencia_max",

    "regressao_linear_min",
    "regressao_linear_max",

    "regressao_ponderada_min",
    "regressao_ponderada_max",

    "media_movel_min",
    "media_movel_max",

]

# ==========================================================
# Ensemble
# ==========================================================

ensemble = [

    "ensemble_min",
    "ensemble_max",

    "ensemble_mediana_min",
    "ensemble_mediana_max",

    "ensemble_desvio_min",
    "ensemble_desvio_max",

    "ensemble_amplitude_min",
    "ensemble_amplitude_max",

    "ensemble_n_modelos_min",
    "ensemble_n_modelos_max",

]

# ==========================================================
# Consolidação das colunas
# ==========================================================

colunas = (

    identificacao

    + historico

    + metricas

    + modelos

    + ensemble

)

# ==========================================================
# Construção da base
# ==========================================================

base_analitica = (

    predicoes[colunas]

    .copy()

)

# ==========================================================
# Organização
# ==========================================================

base_analitica = (

    base_analitica

    .sort_values(

        [

            "curso",
            "campus",
            "turno",
            "modalidade",

        ]

    )

    .reset_index(drop=True)

)

# ==========================================================
# Relatório
# ==========================================================

print(f"\nTotal de ofertas............. {len(base_analitica):,}")

print(f"Total de colunas............ {base_analitica.shape[1]}")

print()

print("Estrutura")

print(f"Identificação............... {len(identificacao)} colunas")
print(f"Histórico................... {len(historico)} colunas")
print(f"Métricas.................... {len(metricas)} colunas")
print(f"Modelos..................... {len(modelos)} colunas")
print(f"Ensemble.................... {len(ensemble)} colunas")

print()

display(base_analitica.head())

print("\nBase analítica construída com sucesso.")

CONSTRUÇÃO DA BASE ANALÍTICA

Total de ofertas............. 765
Total de colunas............ 50

Estrutura
Identificação............... 6 colunas
Histórico................... 6 colunas
Métricas.................... 20 colunas
Modelos..................... 8 colunas
Ensemble.................... 10 colunas



,id_oferta,id_oferta_num,curso,campus,turno,modalidade,observacoes_min,observacoes_max,primeira_nota_min,ultima_nota_min,primeira_nota_max,ultima_nota_max,media_historica_min,mediana_historica_min,minimo_historico_min,maximo_historico_min,amplitude_historica_min,desvio_historico_min,coef_var_min,consistencia_min,mudancas_min,volatilidade_min,media_historica_max,mediana_historica_max,minimo_historico_max,maximo_historico_max,amplitude_historica_max,desvio_historico_max,coef_var_max,consistencia_max,mudancas_max,volatilidade_max,persistencia_min,persistencia_max,regressao_linear_min,regressao_linear_max,regressao_ponderada_min,regressao_ponderada_max,media_movel_min,media_movel_max,ensemble_min,ensemble_max,ensemble_mediana_min,ensemble_mediana_max,ensemble_desvio_min,ensemble_desvio_max,ensemble_amplitude_min,ensemble_amplitude_max,ensemble_n_modelos_min,ensemble_n_modelos_max
0,Administração (Bacharelado) | Darcy Ribeiro | ...,1,Administração (Bacharelado),Darcy Ribeiro,Diurno,AC,6,6,-5.813,38.131,69.035,99.156,26.868,37.418,-5.813,46.485,52.298,23.412,87.136,80.000,2.000,35.681,82.398,96.546,12.807,120.300,107.493,37.802,45.877,60.000,2,65.453,38.131,99.156,37.962,64.876,28.688,62.418,29.510,78.226,33.573,76.169,33.736,71.551,5.177,16.825,9.443,36.738,4,4
1,Administração (Bacharelado) | Darcy Ribeiro | ...,3,Administração (Bacharelado),Darcy Ribeiro,Diurno,CN,6,6,-40.808,-12.580,3.022,6.842,-28.120,-26.694,-54.707,-4.981,49.726,22.874,-81.344,60.000,4.000,48.142,-7.373,1.368,-49.856,44.699,94.555,36.280,-492.069,60.000,3,76.162,-12.580,6.842,-24.194,-10.520,-23.703,-5.991,-27.185,-8.122,-21.916,-4.448,-23.949,-7.057,6.411,7.751,14.605,17.362,4,4
2,Administração (Bacharelado) | Darcy Ribeiro | ...,5,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,4,4,-90.624,-59.221,-3.524,-21.197,-51.749,-50.495,-90.624,-15.383,75.241,31.566,-60.997,66.667,1.000,56.273,-2.144,-1.766,-21.197,16.153,37.350,15.322,-714.546,66.667,1,27.969,-59.221,-21.197,-34.793,-11.358,-53.332,-19.532,-48.358,-3.987,-48.926,-14.019,-50.845,-15.445,10.416,7.950,24.428,17.210,4,4
3,Administração (Bacharelado) | Darcy Ribeiro | ...,7,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPIQ,1,1,-67.831,-67.831,-19.227,-19.227,-67.831,-67.831,-67.831,-67.831,0.000,0.000,0.000,100.000,0.000,0.000,-19.227,-19.227,-19.227,-19.227,0.000,0.000,0.000,100.000,0,0.000,-67.831,-19.227,NaN,NaN,NaN,NaN,-67.831,-19.227,-67.831,-19.227,-67.831,-19.227,0.000,0.000,0.000,0.000,2,2
4,Administração (Bacharelado) | Darcy Ribeiro | ...,8,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_PPI,4,4,-32.289,-53.789,-1.110,-9.772,-42.766,-42.494,-53.789,-32.289,21.500,9.419,-22.024,100.000,0.000,1.148,7.238,7.574,-9.772,23.576,33.348,15.355,212.143,66.667,1,25.647,-53.789,-9.772,-60.975,-1.088,-61.193,-11.231,-46.408,5.573,-55.591,-4.130,-57.382,-5.430,7.022,7.867,14.785,16.804,4,4



Base analítica construída com sucesso.


In [209]:
# ==========================================================
# PARTE 4.6 — INDICADORES DA PREVISÃO
# ==========================================================
#
# Objetivo
# --------
# Construir indicadores derivados diretamente das previsões
# produzidas na Parte 3E.
#
# Nenhum modelo é recalculado.
#
# São produzidos:
#
# • intervalo previsto;
# • largura do intervalo;
# • posição da previsão em relação ao histórico.
# ==========================================================

print("=" * 80)
print("INDICADORES DA PREVISÃO")
print("=" * 80)

# ==========================================================
# Intervalo previsto
# ==========================================================

base_analitica["intervalo_previsto"] = (

    base_analitica["ensemble_max"]

    -

    base_analitica["ensemble_min"]

).round(3)

# ==========================================================
# Diferença entre previsão e média histórica
# ==========================================================

base_analitica["delta_media_min"] = (

    base_analitica["ensemble_min"]

    -

    base_analitica["media_historica_min"]

).round(3)

base_analitica["delta_media_max"] = (

    base_analitica["ensemble_max"]

    -

    base_analitica["media_historica_max"]

).round(3)

# ==========================================================
# Diferença entre previsão e última observação
# ==========================================================

base_analitica["delta_ultima_min"] = (

    base_analitica["ensemble_min"]

    -

    base_analitica["ultima_nota_min"]

).round(3)

base_analitica["delta_ultima_max"] = (

    base_analitica["ensemble_max"]

    -

    base_analitica["ultima_nota_max"]

).round(3)

# ==========================================================
# Resumo estatístico
# ==========================================================

print("\n")
print("=" * 80)
print("RESUMO DOS INDICADORES")
print("=" * 80)

display(

    base_analitica[

        [

            "intervalo_previsto",

            "delta_media_min",
            "delta_media_max",

            "delta_ultima_min",
            "delta_ultima_max",

        ]

    ]

    .describe()

    .round(3)

)

# ==========================================================
# Maiores intervalos previstos
# ==========================================================

print("\n")
print("=" * 80)
print("OFERTAS COM MAIOR INTERVALO PREVISTO")
print("=" * 80)

display(

    base_analitica[

        [

            "curso",
            "campus",
            "turno",
            "modalidade",

            "ensemble_min",
            "ensemble_max",

            "intervalo_previsto",

        ]

    ]

    .sort_values(

        "intervalo_previsto",

        ascending=False

    )

    .head(15)

    .reset_index(drop=True)

)

print("\nIndicadores calculados com sucesso.")

INDICADORES DA PREVISÃO


RESUMO DOS INDICADORES


,intervalo_previsto,delta_media_min,delta_media_max,delta_ultima_min,delta_ultima_max
count,764.000,764.000,765.000,764.000,765.000
mean,26.503,0.617,-2.470,1.189,0.212
std,33.006,18.752,20.609,8.952,10.477
min,-46.928,-71.372,-67.342,-40.226,-45.011
25%,0.000,-4.218,-10.379,-1.554,-2.858
50%,16.144,0.000,0.000,0.000,0.000
75%,46.006,7.675,1.782,4.379,3.382
max,197.299,102.135,94.902,38.948,43.236




OFERTAS COM MAIOR INTERVALO PREVISTO


,curso,campus,turno,modalidade,ensemble_min,ensemble_max,intervalo_previsto
0,Geografia (Bacharelado/Licenciatura),Darcy Ribeiro,Diurno,AC,-76.643,120.656,197.299
1,Física (Licenciatura),Darcy Ribeiro,Noturno,AC,-41.010,136.463,177.473
2,Química Tecnológica (Bacharelado),Darcy Ribeiro,Diurno,AC,-39.332,120.279,159.611
3,Engenharias (FGA),Gama,Diurno,EP_R2_PPI,-43.265,115.838,159.103
4,Geologia (Bacharelado),Darcy Ribeiro,Diurno,AC,-57.183,89.243,146.426
5,Música (Licenciatura),Darcy Ribeiro,Diurno,AC,-18.849,119.527,138.376
6,Física (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_NPPI,-69.660,66.350,136.010
7,Engenharia Química (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_NPPI,-21.089,111.958,133.047
8,Engenharia Florestal (Bacharelado),Darcy Ribeiro,Diurno,AC,-64.123,65.988,130.111
9,História (Licenciatura),Darcy Ribeiro,Noturno,AC,-85.115,37.760,122.875



Indicadores calculados com sucesso.


In [210]:
# ==========================================================
# PARTE 4.7 — VALIDAÇÃO DA BASE ANALÍTICA
# ==========================================================
#
# Objetivo
# --------
# Verificar a consistência da base analítica construída
# nas etapas anteriores.
#
# São verificadas:
#
# • duplicidades;
# • valores ausentes;
# • intervalos inválidos;
# • previsões fora do histórico;
# • inconsistências entre mínimo e máximo.
#
# Esta etapa NÃO modifica a base.
# ==========================================================

print("=" * 80)
print("VALIDAÇÃO DA BASE ANALÍTICA")
print("=" * 80)

# ==========================================================
# Indicadores gerais
# ==========================================================

print("\n")
print("=" * 80)
print("ESTRUTURA")
print("=" * 80)

print(f"Registros...................... {len(base_analitica):,}")
print(f"Colunas........................ {base_analitica.shape[1]}")

duplicadas = base_analitica["id_oferta"].duplicated().sum()

print(f"Duplicidades................... {duplicadas}")

# ==========================================================
# Valores ausentes
# ==========================================================

print("\n")
print("=" * 80)
print("VALORES AUSENTES")
print("=" * 80)

nulos = (

    base_analitica

    .isna()

    .sum()

    .sort_values(ascending=False)

)

nulos = nulos[nulos > 0]

if len(nulos) == 0:

    print("Nenhum valor ausente encontrado.")

else:

    display(

        nulos.rename("Valores ausentes").to_frame()

    )

# ==========================================================
# Intervalos previstos
# ==========================================================

print("\n")
print("=" * 80)
print("CONSISTÊNCIA DOS INTERVALOS")
print("=" * 80)

intervalos_invalidos = (

    base_analitica["ensemble_min"]

    >

    base_analitica["ensemble_max"]

).sum()

print(f"Intervalos inválidos.......... {intervalos_invalidos}")

# ==========================================================
# Valores negativos
# ==========================================================

print("\n")
print("=" * 80)
print("VALORES NEGATIVOS")
print("=" * 80)

negativos = (

    (base_analitica["ensemble_min"] < 0)

    |

    (base_analitica["ensemble_max"] < 0)

).sum()

print(f"Previsões negativas........... {negativos}")

# ==========================================================
# Histórico
# ==========================================================

print("\n")
print("=" * 80)
print("OBSERVAÇÕES HISTÓRICAS")
print("=" * 80)

sem_historico = (

    (base_analitica["observacoes_min"] == 0)

    &

    (base_analitica["observacoes_max"] == 0)

).sum()

print(f"Ofertas sem histórico......... {sem_historico}")

# ==========================================================
# Resumo
# ==========================================================

print("\n")
print("=" * 80)
print("RESUMO DA VALIDAÇÃO")
print("=" * 80)

resumo = pd.DataFrame({

    "Verificação": [

        "Duplicidades",
        "Intervalos inválidos",
        "Previsões negativas",
        "Ofertas sem histórico"

    ],

    "Quantidade": [

        duplicadas,
        intervalos_invalidos,
        negativos,
        sem_historico

    ]

})

display(resumo)

print("\nValidação concluída.")

VALIDAÇÃO DA BASE ANALÍTICA


ESTRUTURA
Registros...................... 765
Colunas........................ 55
Duplicidades................... 0


VALORES AUSENTES


,Valores ausentes
regressao_ponderada_max,270
regressao_ponderada_min,270
regressao_linear_max,270
regressao_linear_min,270
amplitude_historica_min,1
desvio_historico_min,1
minimo_historico_min,1
consistencia_min,1
mudancas_min,1
volatilidade_min,1




CONSISTÊNCIA DOS INTERVALOS
Intervalos inválidos.......... 100


VALORES NEGATIVOS
Previsões negativas........... 561


OBSERVAÇÕES HISTÓRICAS
Ofertas sem histórico......... 0


RESUMO DA VALIDAÇÃO


,Verificação,Quantidade
0,Duplicidades,0
1,Intervalos inválidos,100
2,Previsões negativas,561
3,Ofertas sem histórico,0



Validação concluída.


In [211]:
# ==========================================================
# PARTE 4.8 — COMPONENTES DA ESTABILIDADE HISTÓRICA
# ==========================================================
#
# Objetivo
# --------
# Organizar os principais componentes que descrevem a
# estabilidade histórica das notas mínimas.
#
# Esta etapa NÃO cria um índice composto.
#
# Apenas consolida os indicadores utilizados na etapa
# seguinte.
# ==========================================================

print("=" * 80)
print("COMPONENTES DA ESTABILIDADE HISTÓRICA")
print("=" * 80)

# ==========================================================
# Componentes
# ==========================================================

componentes_estabilidade = [

    "coef_var_min",
    "amplitude_historica_min",
    "volatilidade_min",
    "mudancas_min",
    "observacoes_min",

]

# ==========================================================
# Estatísticas gerais
# ==========================================================

print("\n")
print("=" * 80)
print("ESTATÍSTICAS DOS COMPONENTES")
print("=" * 80)

display(

    base_analitica[
        componentes_estabilidade
    ]

    .describe()

    .round(3)

)

# ==========================================================
# Correlação entre componentes
# ==========================================================

print("\n")
print("=" * 80)
print("CORRELAÇÃO ENTRE OS COMPONENTES")
print("=" * 80)

display(

    base_analitica[
        componentes_estabilidade
    ]

    .corr()

    .round(3)

)

# ==========================================================
# Maiores coeficientes de variação
# ==========================================================

print("\n")
print("=" * 80)
print("MAIOR VARIABILIDADE HISTÓRICA")
print("=" * 80)

display(

    base_analitica[

        [

            "curso",
            "campus",
            "turno",
            "modalidade",

            "coef_var_min",
            "amplitude_historica_min",
            "volatilidade_min",
            "mudancas_min",
            "observacoes_min",

        ]

    ]

    .sort_values(

        "coef_var_min",

        ascending=False

    )

    .head(15)

    .reset_index(drop=True)

)

# ==========================================================
# Maiores amplitudes históricas
# ==========================================================

print("\n")
print("=" * 80)
print("MAIORES AMPLITUDES HISTÓRICAS")
print("=" * 80)

display(

    base_analitica[

        [

            "curso",
            "campus",
            "turno",
            "modalidade",

            "amplitude_historica_min",
            "coef_var_min",
            "volatilidade_min",
            "mudancas_min",

        ]

    ]

    .sort_values(

        "amplitude_historica_min",

        ascending=False

    )

    .head(15)

    .reset_index(drop=True)

)

print("\nComponentes da estabilidade consolidados.")

COMPONENTES DA ESTABILIDADE HISTÓRICA


ESTATÍSTICAS DOS COMPONENTES


,coef_var_min,amplitude_historica_min,volatilidade_min,mudancas_min,observacoes_min
count,764.000,764.000,764.000,764.000,765.000
mean,-28.884,36.832,23.019,0.869,3.008
std,373.754,38.040,27.705,1.079,1.790
min,-4344.623,0.000,0.000,0.000,0.000
25%,-69.468,0.000,0.000,0.000,1.000
50%,0.000,30.234,14.685,0.000,3.000
75%,0.000,62.292,38.753,2.000,5.000
max,3541.599,195.713,147.284,4.000,6.000




CORRELAÇÃO ENTRE OS COMPONENTES


,coef_var_min,amplitude_historica_min,volatilidade_min,mudancas_min,observacoes_min
coef_var_min,1.000,-0.065,-0.112,-0.045,-0.023
amplitude_historica_min,-0.065,1.000,0.890,0.619,0.769
volatilidade_min,-0.112,0.890,1.000,0.655,0.694
mudancas_min,-0.045,0.619,0.655,1.000,0.843
observacoes_min,-0.023,0.769,0.694,0.843,1.000




MAIOR VARIABILIDADE HISTÓRICA


,curso,campus,turno,modalidade,coef_var_min,amplitude_historica_min,volatilidade_min,mudancas_min,observacoes_min
0,Educação Física (Bacharelado),Darcy Ribeiro,Diurno,AC,3541.599,48.851,21.761,0.000,4
1,Fonoaudiologia (Bacharelado),Ceilândia,Diurno,CN,3143.165,75.645,53.680,2.000,5
2,Engenharia Química (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_NPPI,2618.575,94.149,74.173,1.000,4
3,Ciência da Computação (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,2590.027,122.064,68.282,1.000,5
4,Ciências Econômicas (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_NPPI,1870.602,138.683,73.934,1.000,5
5,Direito (Bacharelado),Darcy Ribeiro,Noturno,EP_R1_NPPI,1692.457,57.552,38.856,1.000,5
6,Enfermagem (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,1588.060,52.946,33.160,1.000,5
7,Engenharias (FGA),Gama,Diurno,CN,1416.621,90.495,36.535,2.000,6
8,Engenharia de Computação (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_PPI,1266.773,83.877,61.860,2.000,4
9,Línguas Estrangeiras Aplicadas (Bacharelado),Darcy Ribeiro,Diurno,CN,1105.172,79.054,0.000,0.000,2




MAIORES AMPLITUDES HISTÓRICAS


,curso,campus,turno,modalidade,amplitude_historica_min,coef_var_min,volatilidade_min,mudancas_min
0,Engenharia de Computação (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_NPPI,195.713,216.808,147.284,1.000
1,Engenharia Mecânica (Bacharelado),Darcy Ribeiro,Diurno,AC,190.354,284.823,121.536,4.000
2,Engenharia Elétrica (Bacharelado),Darcy Ribeiro,Diurno,AC,179.938,264.102,146.328,2.000
3,Engenharias (FGA),Gama,Diurno,AC,172.367,249.191,73.662,2.000
4,Engenharia Ambiental (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_PPI,166.758,-775.109,99.338,1.000
5,Engenharia Mecatrônica (Bacharelado),Darcy Ribeiro,Diurno,CN,159.118,106.471,102.668,3.000
6,Engenharia Mecatrônica (Bacharelado),Darcy Ribeiro,Diurno,AC,156.601,131.404,103.770,2.000
7,Física (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_PPI,147.497,622.460,56.862,0.000
8,Design – Programação Visual/Projeto do Produto...,Darcy Ribeiro,Diurno,CN,143.650,609.430,85.194,2.000
9,Física (Bacharelado),Darcy Ribeiro,Diurno,AC,140.899,374.330,91.808,2.000



Componentes da estabilidade consolidados.


In [212]:
# ==========================================================
# PARTE 4.9 — ÍNDICE DE ESTABILIDADE HISTÓRICA
# ==========================================================
#
# Objetivo
# --------
# Construir um índice sintético de estabilidade das notas
# mínimas utilizando exclusivamente o comportamento
# histórico observado.
#
# O índice considera cinco componentes:
#
# • coeficiente de variação;
# • amplitude histórica;
# • volatilidade;
# • mudanças de direção;
# • número de observações.
#
# Todas as componentes recebem o mesmo peso.
#
# O índice varia de 0 a 100.
#
# Quanto MAIOR o índice,
# MAIOR a estabilidade histórica da oferta.
# ==========================================================

print("=" * 80)
print("ÍNDICE DE ESTABILIDADE HISTÓRICA")
print("=" * 80)

# ==========================================================
# Função auxiliar
# ==========================================================

def escore_percentil(serie, crescente=False):
    """
    Converte uma variável em um escore percentual.

    crescente=False
        Menores valores recebem maiores escores.

    crescente=True
        Maiores valores recebem maiores escores.
    """

    if crescente:

        return (

            serie.rank(
                pct=True,
                method="average"
            )

            * 100

        )

    return (

        (1 - serie.rank(
            pct=True,
            method="average"
        ))

        * 100

    )

# ==========================================================
# Componentes padronizadas
# ==========================================================

base_analitica["estab_coef_var"] = escore_percentil(

    base_analitica["coef_var_min"]

)

base_analitica["estab_amplitude"] = escore_percentil(

    base_analitica["amplitude_historica_min"]

)

base_analitica["estab_volatilidade"] = escore_percentil(

    base_analitica["volatilidade_min"]

)

base_analitica["estab_mudancas"] = escore_percentil(

    base_analitica["mudancas_min"]

)

base_analitica["estab_observacoes"] = escore_percentil(

    base_analitica["observacoes_min"],

    crescente=True

)

# ==========================================================
# Índice final
# ==========================================================

componentes = [

    "estab_coef_var",

    "estab_amplitude",

    "estab_volatilidade",

    "estab_mudancas",

    "estab_observacoes",

]

base_analitica["indice_estabilidade"] = (

    base_analitica[componentes]

    .mean(axis=1)

    .round(2)

)

# ==========================================================
# Estatísticas
# ==========================================================

print("\n")
print("=" * 80)
print("ESTATÍSTICAS DO ÍNDICE")
print("=" * 80)

display(

    base_analitica[

        componentes +

        ["indice_estabilidade"]

    ]

    .describe()

    .round(2)

)

# ==========================================================
# Ofertas mais estáveis
# ==========================================================

print("\n")
print("=" * 80)
print("OFERTAS MAIS ESTÁVEIS")
print("=" * 80)

display(

    base_analitica[

        [

            "curso",

            "campus",

            "turno",

            "modalidade",

            "observacoes_min",

            "indice_estabilidade",

        ]

    ]

    .sort_values(

        "indice_estabilidade",

        ascending=False

    )

    .head(15)

    .reset_index(drop=True)

)

# ==========================================================
# Ofertas menos estáveis
# ==========================================================

print("\n")
print("=" * 80)
print("OFERTAS MENOS ESTÁVEIS")
print("=" * 80)

display(

    base_analitica[

        [

            "curso",

            "campus",

            "turno",

            "modalidade",

            "observacoes_min",

            "indice_estabilidade",

        ]

    ]

    .sort_values(

        "indice_estabilidade",

        ascending=True

    )

    .head(15)

    .reset_index(drop=True)

)

print("\nÍndice de estabilidade calculado com sucesso.")

ÍNDICE DE ESTABILIDADE HISTÓRICA


ESTATÍSTICAS DO ÍNDICE


,estab_coef_var,estab_amplitude,estab_volatilidade,estab_mudancas,estab_observacoes,indice_estabilidade
count,764.000,764.000,764.000,764.000,765.000,765.000
mean,49.930,49.930,49.930,49.930,50.070,49.910
std,28.250,28.250,27.680,26.650,28.050,10.370
min,0.000,0.000,0.000,1.510,0.130,0.130
25%,33.380,24.970,24.970,17.210,17.780,42.960
50%,33.380,49.930,49.930,74.540,49.740,54.380
75%,74.900,82.330,78.210,74.540,82.290,57.250
max,99.870,82.330,78.210,74.540,95.290,67.320




OFERTAS MAIS ESTÁVEIS


,curso,campus,turno,modalidade,observacoes_min,indice_estabilidade
0,Engenharia Florestal (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,2,67.320
1,Filosofia (Bacharelado/Licenciatura),Darcy Ribeiro,Diurno,CN,2,67.140
2,Saúde Coletiva (Bacharelado),Darcy Ribeiro,Noturno,EP_R1_NPPI,2,66.900
3,Educação Física Ciclo Básico (Bacharelado/Lice...,Darcy Ribeiro,Diurno,AC,2,66.560
4,Educação Física (Licenciatura),Darcy Ribeiro,Diurno,CN,4,65.940
5,Química Tecnológica (Bacharelado),Darcy Ribeiro,Diurno,CN,2,65.640
6,Química (Licenciatura),Darcy Ribeiro,Noturno,EP_R1_NPPI,2,65.280
7,Comunicação Social – Audiovisual (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,2,65.280
8,Gestão de Agronegócio (Bacharelado),Darcy Ribeiro,Noturno,CN,2,64.700
9,Matemática (Licenciatura),Darcy Ribeiro,Noturno,EP_R2_PPI,2,64.680




OFERTAS MENOS ESTÁVEIS


,curso,campus,turno,modalidade,observacoes_min,indice_estabilidade
0,Biologia (Licenciatura),Darcy Ribeiro,Diurno,EP_R1_PPI,0,0.130
1,Engenharia Ambiental (Bacharelado),Darcy Ribeiro,Diurno,CN,4,18.160
2,Engenharia Mecatrônica (Bacharelado),Darcy Ribeiro,Diurno,CN,5,19.650
3,Engenharia Mecânica (Bacharelado),Darcy Ribeiro,Diurno,AC,6,20.200
4,Engenharia Elétrica (Bacharelado),Darcy Ribeiro,Diurno,AC,5,20.820
5,Engenharia de Computação (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_PPI,4,21.070
6,Física (Bacharelado),Darcy Ribeiro,Diurno,AC,5,21.210
7,Engenharia de Computação (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_NPPI,4,21.480
8,Engenharia Química (Bacharelado),Darcy Ribeiro,Diurno,AC,6,21.660
9,Engenharia Mecatrônica (Bacharelado),Darcy Ribeiro,Diurno,AC,5,21.680



Índice de estabilidade calculado com sucesso.


In [213]:
# ==========================================================
# PARTE 4.10 — CLASSIFICAÇÃO DA ESTABILIDADE HISTÓRICA
# ==========================================================
#
# Objetivo
# --------
# Classificar o Índice de Estabilidade Histórica em cinco
# categorias interpretativas.
#
# A classificação é baseada nos quintis da distribuição do
# índice, produzindo classes balanceadas entre as ofertas.
#
# Classes:
#
# • Muito baixa
# • Baixa
# • Moderada
# • Alta
# • Muito alta
# ==========================================================

print("=" * 80)
print("CLASSIFICAÇÃO DA ESTABILIDADE HISTÓRICA")
print("=" * 80)

# ==========================================================
# Classificação
# ==========================================================

def classificar_estabilidade(indice):

    if pd.isna(indice):
        return np.nan

    if indice >= 80:
        return "Muito alta"

    if indice >= 60:
        return "Alta"

    if indice >= 40:
        return "Moderada"

    if indice >= 20:
        return "Baixa"

    return "Muito baixa"

base_analitica["classe_estabilidade"] = (

    base_analitica["indice_estabilidade"]

    .apply(classificar_estabilidade)

)

# ==========================================================
# Distribuição das classes
# ==========================================================

print("\n")
print("=" * 80)
print("DISTRIBUIÇÃO DAS CLASSES")
print("=" * 80)

distribuicao = (

    base_analitica["classe_estabilidade"]

    .value_counts()

    .reindex(rotulos)

    .rename_axis("Classe")

    .reset_index(name="Ofertas")

)

distribuicao["Percentual (%)"] = (

    distribuicao["Ofertas"]

    /

    len(base_analitica)

    * 100

).round(2)

display(distribuicao)

# ==========================================================
# Estatísticas por classe
# ==========================================================

print("\n")
print("=" * 80)
print("ESTATÍSTICAS POR CLASSE")
print("=" * 80)

display(

    base_analitica

    .groupby("classe_estabilidade", observed=False)[

        [

            "indice_estabilidade",

            "coef_var_min",

            "amplitude_historica_min",

            "volatilidade_min",

            "mudancas_min",

            "observacoes_min",

        ]

    ]

    .mean()

    .round(2)

)

# ==========================================================
# Exemplos de maior estabilidade
# ==========================================================

print("\n")
print("=" * 80)
print("OFERTAS MAIS ESTÁVEIS")
print("=" * 80)

display(

    base_analitica[

        [

            "curso",

            "campus",

            "turno",

            "modalidade",

            "indice_estabilidade",

            "classe_estabilidade",

            "observacoes_min",

            "coef_var_min",

            "amplitude_historica_min",

        ]

    ]

    .sort_values(

        "indice_estabilidade",

        ascending=False

    )

    .head(15)

    .reset_index(drop=True)

)

# ==========================================================
# Exemplos de menor estabilidade
# ==========================================================

print("\n")
print("=" * 80)
print("OFERTAS MENOS ESTÁVEIS")
print("=" * 80)

display(

    base_analitica[

        [

            "curso",

            "campus",

            "turno",

            "modalidade",

            "indice_estabilidade",

            "classe_estabilidade",

            "observacoes_min",

            "coef_var_min",

            "amplitude_historica_min",

        ]

    ]

    .sort_values(

        "indice_estabilidade",

        ascending=True

    )

    .head(15)

    .reset_index(drop=True)

)

print("\nClassificação da estabilidade concluída.")

CLASSIFICAÇÃO DA ESTABILIDADE HISTÓRICA


DISTRIBUIÇÃO DAS CLASSES


,Classe,Ofertas,Percentual (%)
0,Muito baixa,3.000,0.390
1,Baixa,128.000,16.730
2,Moderada,569.000,74.380
3,Alta,65.000,8.500
4,Muito alta,NaN,NaN




ESTATÍSTICAS POR CLASSE


,indice_estabilidade,coef_var_min,amplitude_historica_min,volatilidade_min,mudancas_min,observacoes_min
classe_estabilidade,,,,,,
Alta,62.710,-86.410,23.600,1.260,0.000,2.310
Baixa,32.590,151.980,82.390,59.130,2.040,4.840
Moderada,52.540,-63.710,27.730,17.120,0.700,2.680
Muito baixa,12.650,172.690,141.460,98.730,2.500,3.000




OFERTAS MAIS ESTÁVEIS


,curso,campus,turno,modalidade,indice_estabilidade,classe_estabilidade,observacoes_min,coef_var_min,amplitude_historica_min
0,Engenharia Florestal (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,67.320,Alta,2,-105.998,13.244
1,Filosofia (Bacharelado/Licenciatura),Darcy Ribeiro,Diurno,CN,67.140,Alta,2,-278.518,31.302
2,Saúde Coletiva (Bacharelado),Darcy Ribeiro,Noturno,EP_R1_NPPI,66.900,Alta,2,-84.303,4.106
3,Educação Física Ciclo Básico (Bacharelado/Lice...,Darcy Ribeiro,Diurno,AC,66.560,Alta,2,-159.308,28.661
4,Educação Física (Licenciatura),Darcy Ribeiro,Diurno,CN,65.940,Alta,4,-345.296,40.714
5,Química Tecnológica (Bacharelado),Darcy Ribeiro,Diurno,CN,65.640,Alta,2,-123.531,30.064
6,Química (Licenciatura),Darcy Ribeiro,Noturno,EP_R1_NPPI,65.280,Alta,2,-358.063,44.265
7,Comunicação Social – Audiovisual (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,65.280,Alta,2,-223.267,41.786
8,Gestão de Agronegócio (Bacharelado),Darcy Ribeiro,Noturno,CN,64.700,Alta,2,-50.860,13.501
9,Matemática (Licenciatura),Darcy Ribeiro,Noturno,EP_R2_PPI,64.680,Alta,2,-69.264,22.030




OFERTAS MENOS ESTÁVEIS


,curso,campus,turno,modalidade,indice_estabilidade,classe_estabilidade,observacoes_min,coef_var_min,amplitude_historica_min
0,Biologia (Licenciatura),Darcy Ribeiro,Diurno,EP_R1_PPI,0.130,Muito baixa,0,NaN,NaN
1,Engenharia Ambiental (Bacharelado),Darcy Ribeiro,Diurno,CN,18.160,Muito baixa,4,238.903,123.811
2,Engenharia Mecatrônica (Bacharelado),Darcy Ribeiro,Diurno,CN,19.650,Muito baixa,5,106.471,159.118
3,Engenharia Mecânica (Bacharelado),Darcy Ribeiro,Diurno,AC,20.200,Baixa,6,284.823,190.354
4,Engenharia Elétrica (Bacharelado),Darcy Ribeiro,Diurno,AC,20.820,Baixa,5,264.102,179.938
5,Engenharia de Computação (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_PPI,21.070,Baixa,4,1266.773,83.877
6,Física (Bacharelado),Darcy Ribeiro,Diurno,AC,21.210,Baixa,5,374.330,140.899
7,Engenharia de Computação (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_NPPI,21.480,Baixa,4,216.808,195.713
8,Engenharia Química (Bacharelado),Darcy Ribeiro,Diurno,AC,21.660,Baixa,6,197.837,120.196
9,Engenharia Mecatrônica (Bacharelado),Darcy Ribeiro,Diurno,AC,21.680,Baixa,5,131.404,156.601



Classificação da estabilidade concluída.


In [214]:
# ==========================================================
# PARTE 4.11 — PREPARAÇÃO DA PLANILHA DO ESTUDANTE
# ==========================================================
#
# Objetivo
# --------
# Construir a base final que será exportada para a planilha
# destinada aos estudantes.
#
# A planilha privilegia a nota mínima, pois representa o
# principal parâmetro de aprovação.
#
# São incluídos:
#
# • identificação da oferta;
# • histórico recente;
# • previsão do ensemble;
# • variação em relação ao histórico;
# • estabilidade histórica;
# • informações auxiliares.
# ==========================================================

print("=" * 80)
print("PREPARAÇÃO DA PLANILHA DO ESTUDANTE")
print("=" * 80)

# ==========================================================
# Seleção das colunas
# ==========================================================

colunas_planilha = [

    # ------------------------------------------------------
    # Identificação
    # ------------------------------------------------------

    "curso",
    "campus",
    "turno",
    "modalidade",

    # ------------------------------------------------------
    # Histórico
    # ------------------------------------------------------

    "observacoes_min",

    "primeira_nota_min",
    "ultima_nota_min",

    "media_historica_min",
    "mediana_historica_min",

    "minimo_historico_min",
    "maximo_historico_min",

    # ------------------------------------------------------
    # Previsão
    # ------------------------------------------------------

    "ensemble_min",

    "delta_media_min",

    "delta_ultima_min",

    # ------------------------------------------------------
    # Estabilidade
    # ------------------------------------------------------

    "indice_estabilidade",

    "classe_estabilidade",

    "coef_var_min",

    "desvio_historico_min",

    "amplitude_historica_min",

    "volatilidade_min",

    "mudancas_min",

]

# ==========================================================
# Construção
# ==========================================================

planilha_final = (

    base_analitica[colunas_planilha]

    .copy()

)

# ==========================================================
# Renomeação
# ==========================================================

planilha_final.rename(

    columns={

        "curso":
            "Curso",

        "campus":
            "Campus",

        "turno":
            "Turno",

        "modalidade":
            "Modalidade",

        "observacoes_min":
            "Triênios",

        "primeira_nota_min":
            "Primeira Nota",

        "ultima_nota_min":
            "Última Nota",

        "media_historica_min":
            "Média Histórica",

        "mediana_historica_min":
            "Mediana Histórica",

        "minimo_historico_min":
            "Mínimo Histórico",

        "maximo_historico_min":
            "Máximo Histórico",

        "ensemble_min":
            "Previsão",

        "delta_media_min":
            "Δ Média",

        "delta_ultima_min":
            "Δ Última",

        "indice_estabilidade":
            "Índice de Estabilidade",

        "classe_estabilidade":
            "Estabilidade",

        "coef_var_min":
            "CV (%)",

        "desvio_historico_min":
            "Desvio",

        "amplitude_historica_min":
            "Amplitude",

        "volatilidade_min":
            "Volatilidade",

        "mudancas_min":
            "Mudanças",

    },

    inplace=True

)

# ==========================================================
# Organização
# ==========================================================

planilha_final = (

    planilha_final

    .sort_values(

        [

            "Curso",

            "Campus",

            "Turno",

            "Modalidade",

        ]

    )

    .reset_index(drop=True)

)

# ==========================================================
# Arredondamento
# ==========================================================

colunas_numericas = planilha_final.select_dtypes(

    include="number"

).columns

planilha_final[colunas_numericas] = (

    planilha_final[colunas_numericas]

    .round(2)

)

# ==========================================================
# Relatório
# ==========================================================

print(f"\nTotal de ofertas............. {len(planilha_final):,}")

print(f"Total de colunas............ {planilha_final.shape[1]}")

print()

print("Colunas da planilha")

for i, coluna in enumerate(planilha_final.columns, start=1):

    print(f"{i:02d} - {coluna}")

print()

display(planilha_final.head(10))

print("\nPlanilha preparada com sucesso.")

PREPARAÇÃO DA PLANILHA DO ESTUDANTE

Total de ofertas............. 765
Total de colunas............ 21

Colunas da planilha
01 - Curso
02 - Campus
03 - Turno
04 - Modalidade
05 - Triênios
06 - Primeira Nota
07 - Última Nota
08 - Média Histórica
09 - Mediana Histórica
10 - Mínimo Histórico
11 - Máximo Histórico
12 - Previsão
13 - Δ Média
14 - Δ Última
15 - Índice de Estabilidade
16 - Estabilidade
17 - CV (%)
18 - Desvio
19 - Amplitude
20 - Volatilidade
21 - Mudanças



,Curso,Campus,Turno,Modalidade,Triênios,Primeira Nota,Última Nota,Média Histórica,Mediana Histórica,Mínimo Histórico,Máximo Histórico,Previsão,Δ Média,Δ Última,Índice de Estabilidade,Estabilidade,CV (%),Desvio,Amplitude,Volatilidade,Mudanças
0,Administração (Bacharelado),Darcy Ribeiro,Diurno,AC,6,-5.810,38.130,26.870,37.420,-5.810,46.480,33.570,6.700,-4.560,36.220,Baixa,87.140,23.410,52.300,35.680,2.000
1,Administração (Bacharelado),Darcy Ribeiro,Diurno,CN,6,-40.810,-12.580,-28.120,-26.690,-54.710,-4.980,-21.920,6.200,-9.340,45.410,Moderada,-81.340,22.870,49.730,48.140,4.000
2,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,4,-90.620,-59.220,-51.750,-50.500,-90.620,-15.380,-48.930,2.820,10.300,41.210,Moderada,-61.000,31.570,75.240,56.270,1.000
3,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPIQ,1,-67.830,-67.830,-67.830,-67.830,-67.830,-67.830,-67.830,0.000,0.000,57.250,Moderada,0.000,0.000,0.000,0.000,0.000
4,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_PPI,4,-32.290,-53.790,-42.770,-42.490,-53.790,-32.290,-55.590,-12.820,-1.800,61.910,Alta,-22.020,9.420,21.500,1.150,0.000
5,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_NPPI,5,-80.470,-77.480,-31.060,-11.330,-80.470,13.600,-45.490,-14.430,31.990,44.630,Moderada,-143.680,44.630,94.070,66.800,1.000
6,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_NPPIQ,1,-17.430,-17.430,-17.430,-17.430,-17.430,-17.430,-17.430,0.000,0.000,57.250,Moderada,0.000,0.000,0.000,0.000,0.000
7,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_PPI,4,-66.920,-51.290,-41.330,-42.100,-66.920,-14.210,-38.360,2.980,12.930,46.450,Moderada,-55.170,22.800,52.710,37.410,1.000
8,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R2_PPIQ,1,-25.420,-25.420,-25.420,-25.420,-25.420,-25.420,-25.420,0.000,0.000,57.250,Moderada,0.000,0.000,0.000,0.000,0.000
9,Administração (Bacharelado),Darcy Ribeiro,Noturno,AC,6,-98.900,-7.050,-38.010,-16.270,-98.900,-2.780,-21.730,16.270,-14.680,42.210,Moderada,-113.060,42.970,96.120,68.730,2.000



Planilha preparada com sucesso.


In [222]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# ==============================================================================
# PARTE 4.11A — BASE HISTÓRICA PARA INTELIGÊNCIA ANALÍTICA
# ==============================================================================
#
# Objetivo
# --------
# Construir uma base contendo a representação completa das
# séries temporais de cada oferta.
#
# Esta base NÃO recalcula modelos.
#
# Apenas organiza informações temporais que serão utilizadas
# pelo Notebook 05 na construção dos Scores Analíticos.
#
# Arquivo produzido:
#
# historico_ofertas.csv
#
# ==============================================================================

print("=" * 80)
print("BASE HISTÓRICA PARA INTELIGÊNCIA ANALÍTICA")
print("=" * 80)

registros = []

for oferta in dict_series.values():

    # x_min and y_min are already prepared in dict_series
    x_min = oferta["x_min"]
    y_min = np.asarray(oferta["y_min"], dtype=float)

    # x_max and y_max are already prepared in dict_series
    x_max = oferta["x_max"]
    y_max = np.asarray(oferta["y_max"], dtype=float)

    # Use pesos_min for y_min and pesos_max for y_max
    pesos_min_serie = oferta["pesos_min"]
    pesos_max_serie = oferta["pesos_max"]

    # --------------------------------------------------------------------------
    # Médias ponderadas
    # --------------------------------------------------------------------------
    if len(y_min) > 0:
        media_pond_min = np.average(
            y_min,
            weights=pesos_min_serie
        )
    else:
        media_pond_min = np.nan

    if len(y_max) > 0:
        media_pond_max = np.average(
            y_max,
            weights=pesos_max_serie
        )
    else:
        media_pond_max = np.nan

    # --------------------------------------------------------------------------
    # Desvio ponderado
    # --------------------------------------------------------------------------
    if len(y_min) > 0:
        desvio_pond_min = np.sqrt(
            np.average(
                (y_min - media_pond_min) ** 2,
                weights=pesos_min_serie
            )
        )
    else:
        desvio_pond_min = np.nan

    if len(y_max) > 0:
        desvio_pond_max = np.sqrt(
            np.average(
                (y_max - media_pond_max) ** 2,
                weights=pesos_max_serie
            )
        )
    else:
        desvio_pond_max = np.nan

    # --------------------------------------------------------------------------
    # Tendência linear ponderada
    # --------------------------------------------------------------------------

    modelo = LinearRegression()

    # Fit for y_min
    if len(y_min) >= 2:
        modelo.fit(
            x_min,
            y_min,
            sample_weight=pesos_min_serie
        )
        slope_ponderada_min = modelo.coef_[0]
        r2_ponderada_min = modelo.score(
            x_min,
            y_min,
            sample_weight=pesos_min_serie
        )
    else:
        slope_ponderada_min = np.nan
        r2_ponderada_min = np.nan

    # Fit for y_max
    if len(y_max) >= 2:
        modelo.fit(
            x_max,
            y_max,
            sample_weight=pesos_max_serie
        )
        slope_ponderada_max = modelo.coef_[0]
        r2_ponderada_max = modelo.score(
            x_max,
            y_max,
            sample_weight=pesos_max_serie
        )
    else:
        slope_ponderada_max = np.nan
        r2_ponderada_max = np.nan

    # --------------------------------------------------------------------------
    # Momentum
    #
    # Diferença entre a média ponderada e a média simples.
    #
    # Valores positivos indicam crescimento recente.
    # --------------------------------------------------------------------------
    if len(y_min) > 0:
        momentum_min = (
            media_pond_min
            - np.mean(y_min)
        )
    else:
        momentum_min = np.nan

    if len(y_max) > 0:
        momentum_max = (
            media_pond_max
            - np.mean(y_max)
        )
    else:
        momentum_max = np.nan

    # --------------------------------------------------------------------------
    # Crescimento total da série
    # --------------------------------------------------------------------------

    delta_total_min = y_min[-1] - y_min[0] if len(y_min) > 0 else np.nan

    delta_total_max = y_max[-1] - y_max[0] if len(y_max) > 0 else np.nan

    # --------------------------------------------------------------------------
    # Crescimento percentual
    # --------------------------------------------------------------------------

    if len(y_min) > 0 and y_min[0] != 0:

        crescimento_percentual_min = (
            100 * delta_total_min / y_min[0]
        )

    else:

        crescimento_percentual_min = np.nan

    if len(y_max) > 0 and y_max[0] != 0:

        crescimento_percentual_max = (
            100 * delta_total_max / y_max[0]
        )

    else:

        crescimento_percentual_max = np.nan

    # --------------------------------------------------------------------------
    # Aceleração
    #
    # Segunda diferença da série.
    # --------------------------------------------------------------------------

    aceleracao_min = (
        np.mean(np.diff(y_min, n=2))
        if len(y_min) >= 3
        else np.nan
    )

    aceleracao_max = (
        np.mean(np.diff(y_max, n=2))
        if len(y_max) >= 3
        else np.nan
    )

    # --------------------------------------------------------------------------
    # Curvatura
    #
    # Ajuste polinomial de grau 2.
    # --------------------------------------------------------------------------

    if len(y_min) >= 3:

        coef = np.polyfit(
            x_min.flatten(), # polyfit expects 1D array
            y_min,
            deg=2
        )

        curvatura_min = coef[0]

    else:

        curvatura_min = np.nan

    if len(y_max) >= 3:

        coef = np.polyfit(
            x_max.flatten(), # polyfit expects 1D array
            y_max,
            deg=2
        )

        curvatura_max = coef[0]

    else:

        curvatura_max = np.nan

    # --------------------------------------------------------------------------
    # Registro
    # --------------------------------------------------------------------------

    registros.append({

        "id_oferta":
            oferta["id_oferta"],

        "id_oferta_num":
            oferta["id_oferta_num"],

        "curso":
            oferta["curso"],

        "campus":
            oferta["campus"],

        "turno":
            oferta["turno"],

        "modalidade":
            oferta["modalidade"],

        "n_trienios":
            len(y_min),

        "media_ponderada_min":
            round(media_pond_min,3) if pd.notna(media_pond_min) else np.nan,

        "media_ponderada_max":
            round(media_pond_max,3) if pd.notna(media_pond_max) else np.nan,

        "desvio_ponderado_min":
            round(desvio_pond_min,3) if pd.notna(desvio_pond_min) else np.nan,

        "desvio_ponderado_max":
            round(desvio_pond_max,3) if pd.notna(desvio_pond_max) else np.nan,

        "slope_ponderada_min":
            round(slope_ponderada_min,5) if pd.notna(slope_ponderada_min) else np.nan,

        "slope_ponderada_max":
            round(slope_ponderada_max,5) if pd.notna(slope_ponderada_max) else np.nan,

        "momentum_min":
            round(momentum_min,3) if pd.notna(momentum_min) else np.nan,

        "momentum_max":
            round(momentum_max,3) if pd.notna(momentum_max) else np.nan,

        "aceleracao_min":
            round(aceleracao_min,5)
            if pd.notna(aceleracao_min)
            else np.nan,

        "aceleracao_max":
            round(aceleracao_max,5)
            if pd.notna(aceleracao_max)
            else np.nan,

        "curvatura_min":
            round(curvatura_min,6)
            if pd.notna(curvatura_min)
            else np.nan,

        "curvatura_max":
            round(curvatura_max,6)
            if pd.notna(curvatura_max)
            else np.nan,

    })

historico_ofertas = (
    pd.DataFrame(registros)
    .sort_values("id_oferta_num")
    .reset_index(drop=True)
)

# ==============================================================================
# RELATÓRIO
# ==============================================================================

print(f"\nOfertas..................... {len(historico_ofertas):,}")

print(f"Colunas..................... {historico_ofertas.shape[1]}")

display(historico_ofertas.head())

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

salvar_csv(
    historico_ofertas,
    "historico_ofertas.csv"
)

salvar_parquet(
    historico_ofertas,
    "historico_ofertas.parquet"
)

print("\nArquivos produzidos")

print("✓ historico_ofertas.csv")

print("✓ historico_ofertas.parquet")

BASE HISTÓRICA PARA INTELIGÊNCIA ANALÍTICA

Ofertas..................... 765
Colunas..................... 19


,id_oferta,id_oferta_num,curso,campus,turno,modalidade,n_trienios,media_ponderada_min,media_ponderada_max,desvio_ponderado_min,desvio_ponderado_max,slope_ponderada_min,slope_ponderada_max,momentum_min,momentum_max,aceleracao_min,aceleracao_max,curvatura_min,curvatura_max
0,Administração (Bacharelado) | Darcy Ribeiro | ...,1,Administração (Bacharelado),Darcy Ribeiro,Diurno,AC,6,29.510,78.226,19.563,37.771,-0.308,-5.928,2.641,-4.172,-1.131,8.771,-4.347,-1.152
1,Administração (Bacharelado) | Darcy Ribeiro | ...,2,Administração (Bacharelado),Darcy Ribeiro,Noturno,AC,6,-33.119,31.431,35.588,42.976,1.436,-11.672,4.887,-9.171,-4.101,6.578,-5.536,-0.834
2,Administração (Bacharelado) | Darcy Ribeiro | ...,3,Administração (Bacharelado),Darcy Ribeiro,Diurno,CN,6,-27.185,-8.122,21.473,35.317,1.306,0.799,0.935,-0.749,1.776,14.702,0.230,2.123
3,Administração (Bacharelado) | Darcy Ribeiro | ...,4,Administração (Bacharelado),Darcy Ribeiro,Noturno,CN,5,-30.223,-14.281,24.094,28.918,-9.639,-15.208,-7.163,-7.627,-16.976,-17.645,-0.549,-5.580
4,Administração (Bacharelado) | Darcy Ribeiro | ...,5,Administração (Bacharelado),Darcy Ribeiro,Diurno,EP_R1_NPPI,4,-48.358,-3.987,21.362,15.600,-2.487,-7.772,3.391,-1.843,-46.346,-20.433,-23.173,-10.216



Arquivos produzidos
✓ historico_ofertas.csv
✓ historico_ofertas.parquet


In [223]:
# ==========================================================
# PARTE 4.12 — EXPORTAÇÃO DOS RESULTADOS
# ==========================================================
#
# Objetivo
# --------
# Exportar os principais produtos gerados pelo Notebook 04.
#
# Arquivos produzidos:
#
# • predicoes.csv
# • base_analitica.csv
# • planilha_estudante.csv
# • previsoes_pas_unb.xlsx
#
# Todos os arquivos são gravados na pasta de saída do
# projeto definida na configuração do notebook.
# ==========================================================

print("=" * 80)
print("EXPORTAÇÃO DOS RESULTADOS")
print("=" * 80)

# ==========================================================
# Arquivos CSV
# ==========================================================

arquivos = {

    "predicoes.csv":
        predicoes,

    "base_analitica.csv":
        base_analitica,

    "planilha_estudante.csv":
        planilha_final,

}

for nome, df in arquivos.items():

    caminho = PASTA_SAIDA / nome

    df.to_csv(

        caminho,

        index=False,

        encoding="utf-8-sig",

    )

# ==========================================================
# Arquivo Excel
# ==========================================================

arquivo_excel = (

    PASTA_SAIDA

    / "previsoes_pas_unb.xlsx"

)

with pd.ExcelWriter(

    arquivo_excel,

    engine="openpyxl",

) as writer:

    predicoes.to_excel(

        writer,

        sheet_name="Predicoes",

        index=False,

    )

    base_analitica.to_excel(

        writer,

        sheet_name="Base_Analitica",

        index=False,

    )

    planilha_final.to_excel(

        writer,

        sheet_name="Planilha_Estudante",

        index=False,

    )

# ==========================================================
# Relatório
# ==========================================================

print()

print(f"Diretório de saída:")

print(PASTA_SAIDA)

print()

print("Arquivos exportados:")

for nome in arquivos.keys():

    caminho = PASTA_SAIDA / nome

    tamanho = caminho.stat().st_size / 1024

    print(f"✓ {nome:<30} {tamanho:8.1f} KB")

caminho = PASTA_SAIDA / "previsoes_pas_unb.xlsx"

tamanho = caminho.stat().st_size / 1024

print(f"✓ {'previsoes_pas_unb.xlsx':<30} {tamanho:8.1f} KB")

print()

print("=" * 80)
print("EXPORTAÇÃO CONCLUÍDA")
print("=" * 80)

print(f"Total de ofertas............. {len(predicoes):,}")

print(f"Base analítica............... {len(base_analitica):,}")

print(f"Planilha do estudante........ {len(planilha_final):,}")

print()

print("Todos os resultados foram gravados com sucesso.")

EXPORTAÇÃO DOS RESULTADOS

Diretório de saída:
/content/drive/MyDrive/Dados/PAS/estatisticas

Arquivos exportados:
✓ predicoes.csv                     352.3 KB
✓ base_analitica.csv                437.1 KB
✓ planilha_estudante.csv            120.9 KB
✓ previsoes_pas_unb.xlsx            546.9 KB

EXPORTAÇÃO CONCLUÍDA
Total de ofertas............. 765
Base analítica............... 765
Planilha do estudante........ 765

Todos os resultados foram gravados com sucesso.
